# 한국 지역별(시도/시군구) HS 수출 수집 · 예측 · 기업 매출 연동 — v7

> **v7**: 2단계가 전부 끝나기 전이라도, 과거 자료가 다 찬 계열(품질 A/B)부터 매일 예측한다. 같은 달에 이미 예측한 계열은 다시 하지 않는다.
> 1단계 구간을 2025-09~2026-08 로 고정(SCAN_WINDOW_END="202608")해, 달이 바뀌어도 1단계를 다시 받지 않는다.
>
> **사용법: 매일 한 번 "전체 실행(Run All)" 하면 끝입니다.** 맨 아래 PART 10 이 남은 일을 알아서 합니다.
> 목록 갱신 -> 1단계 수집 -> 2단계 과거 채우기 -> 품질 등급 -> 예측. 하루 호출 한도에 닿으면 멈추고, 다음 날 이어서 합니다.
> 손댈 설정은 없습니다. (PART 8-0 의 DAILY_CALL_LIMIT 만 실제 한도와 맞추면 됩니다)

전국 HS 수출 대신 **기업 소재 지역의 HS 수출**로 기업 매출을 추정하기 위한 파이프라인입니다.
입력은 `(ticker, 기업명, HS코드, 시도[, 시군구])` 한 줄이면 됩니다. 예: `("A080220", "제주반도체", "854232", "제주")`

| PART | 내용 | 산출 |
|---|---|---|
| 1 | 환경 · Control Panel | |
| 2 | 시도코드 사전, 관세청 API 호출/파싱, `probe_api()` | |
| 3 | 응답 표준화 → 월별 long, DB 업서트, 증분 수집, 시계열 로드 | `korea_regional_trade_data` |
| 4 | SARIMA/ETS/Theta + median 앙상블 예측 (무실적 월이 많으면 분기 예측) | `korea_regional_trade_forecast` (forecast_date vintage 보존) |
| 5 | 기업-지역-HS 매핑 저장 | `korea_company_region_hscode_map` |
| 6 | 매출 실적/시계열예측 로드, 연동성 검정, 수출함의 매출(M1 비율법, M2 탄력성) | |
| 7 | 실행 (probe → 수집 → 예측 → 분석 → Excel) | `한국_지역별수출분석/YYYYMM/regional_export_revenue_YYYYMMDD.xlsx` |

### v2 → v3 변경
| # | 항목 | 내용 |
|---|---|---|
| 1 | **12개월 초과 기간 자동 분할** | API는 조회기간이 12개월을 넘으면 데이터를 반환하지 않음. `fetch_window()`가 요청 기간을 12개월 이하 구간으로 자동 분할·순차 호출 후 결합하도록 변경 → `probe_api`, 스캔 셀, 수집 파이프라인 등 **어디서 호출해도** 기간 제한 없음 (예: 201701~202608 → 10개 구간) |
| 2 | **구간별 로그** | 구간마다 resultCode/행수를 기록, 비정상 구간은 경고 출력 및 Collect_Log에 남김 |
| 3 | **단일 요청 가드** | 내부 함수 `_fetch_single_window()`에 12개월 초과 요청이 들어오면 즉시 오류 (조용한 누락 방지) |

### v1 → v2 변경
| # | 항목 | 내용 |
|---|---|---|
| 1 | **금액 단위 보정** | API 금액(`expUsdAmt` 등)은 **천 달러** 단위로 확인됨(제주 2026.1~8 반도체 합계 471,191 ≈ 뉴스 4.7억 달러). `API_AMOUNT_UNIT = 1000`으로 달러 환산해 전국 테이블(달러)과 단위 통일 |
| 2 | **복수 HS 입력** | `"854239+854231"`처럼 입력하면 HS별 개별 분석 + 합산 시계열(재예측) 분석을 함께 산출 |
| 3 | **HS 자릿수** | API는 6자리만 지원(2/4자리 요청은 resultCode 99) → 입력 검증에 반영 |
| 4 | **응답 필드 확정** | `priodTitle`, `sggNm`, `hsSgn`, `expUsdAmt`, `impUsdAmt`, `expCnt`, `impCnt`, `cmtrBlncAmt` — 요청 파라미터명(`strtYymm/endYymm/sidoCd/hsSgn`)도 정상 확인 |

> **v1으로 이미 수집을 실행했다면**: DB에 천 달러 단위 값이 들어가 있습니다. v2에서 `INCREMENTAL = False`로 **한 번 전체 재수집**하면 같은 키가 달러 단위로 덮어써집니다. 이후 `True`로 되돌리세요.

### 반드시 먼저 확인할 것
1. API 파라미터·필드는 probe로 검증 완료(v2). 응답 형식이 바뀌면 `API_PARAM_NAMES`(PART 1-2) / `FIELD_CANDIDATES`(PART 3-1)만 수정하세요.
2. **광주·전남 통합**: 2026-07부터 시도코드 12(전남광주통합특별시). 29(광주)·46(전남)은 2026-06까지만 존재합니다. 해당 지역 기업은 **'12'로 입력**하면 29+46 합산 → 12로 이어 붙인 연속 시계열이 자동 생성됩니다.
3. **지역 귀속 기준**: 지역별 무역통계는 일반적으로 수출 화주(신고자)의 사업장 소재지 기준으로 집계됩니다. 생산공장이 아닌 본사·수출법인 주소로 잡힐 수 있으니, 매핑 지역은 **실제 수출신고 주체의 소재지**로 넣는 것이 정확합니다.
4. 수집 데이터는 `ON DUPLICATE KEY UPDATE`로 저장합니다. 최근 `REVISION_MONTHS`개월은 매번 재수집해 확정치 수정분을 반영합니다(기존 전국 수집기의 '신규 키만 추가' 방식과 다름).

### 결과 해석 (Summary 시트)
- `grade`: 지역수출-매출 연동성 A(검증)/B(약함)/C(무효). A = HAC p<0.05, β>0, 표본외 RMSE < naive, Pearson-Spearman 괴리 ≤0.3, 점유율 드리프트 없음
- `g_rev_TS_12`: 기존 매출 시계열 예측(수출 정보 미사용) 기준 향후 4분기 성장률
- `g_rev_M1_ratio_12`: 최근 4분기 (매출 ÷ 원화환산 지역수출) 비율 × 미래 지역수출 — `ratio_cv_8q`가 낮을수록 신뢰
- `g_rev_M2_elastic_12`: YoY 탄력성(β, 최적 lag) 적용 — 등급 A/B일 때만 산출
- `region_share_of_national_12m`: 해당 지역이 전국 동일 HS 수출에서 차지하는 비중


### v4 -> v5 변경 (지역 수출 예측)
| # | 항목 | 내용 |
|---|---|---|
| 1 | **PART 9 예측** | 품질 등급 A(월)·B(분기) 계열을 SARIMA(후보 8개 차수 중 AICc 최소) + ETS(감쇠 추세·가법 계절) + Theta 로 예측, 앙상블(중앙값 또는 검증 오차 역수 가중) |
| 2 | **검증** | 마지막 12개월(분기 4개)을 떼고 다시 적합해 모형별·앙상블 sMAPE 와 작년 같은 달(단순 기준) sMAPE 를 함께 저장 |
| 3 | **DB 넓은 형식** | `korea_regional_fc_run`(계열 x 예측일 1행: 차수, 오차, 가중치, 상태) + `korea_regional_fc_value`(대상월 1행에 앙상블·SARIMA·ETS·Theta 열). 기존 긴 형식보다 행 수 4분의 1 |
| 4 | **최신 예측 뷰** | `korea_regional_fc_latest`: 계열별 가장 최근 예측일 값만 |
| 5 | **이어서 실행** | 같은 예측일에 끝난 계열은 건너뜀, 100계열마다 저장. `FC_N_JOBS` 로 병렬 |
| 6 | HS6 목록 CSV | `HS6_MASTER_CSV` 경로가 틀리면 경고 출력 (v4 는 조용히 무시) |

PART 8-7(v3 방식 선택 예측)은 남겨 두었지만, 앞으로는 PART 9 를 씁니다.

### v3 -> v4 변경 (대량 수집 3단계)
API는 시도와 HS(6자리)가 모두 필수이므로 (시도 x HS x 12개월 구간) 단위로만 받을 수 있습니다. 그래서 전부 받는 대신 3단계로 받습니다.

| # | 항목 | 내용 |
|---|---|---|
| 1 | **대상 HS 목록** (PART 8-2) | 전국 수집기 HS_CODE_MAP, kr_company_hs_map, fd_driver_map_2d, 지역 매핑, 수동 후보를 합치고 4자리 코드는 6자리 하위 코드로 펼침 -> `korea_regional_hs_universe` |
| 2 | **1단계 지도** (PART 8-3) | 모든 (시도 x HS)를 최근 12개월 1구간만 요청 -> `korea_regional_trade_scan` (0인 조합도 기록, 다시 요청 안 함) |
| 3 | **2단계 과거 채우기** (PART 8-4) | 수출 있는 조합만 START_YYYYMM 까지. 지역 매핑 조합 -> 기업 연결 HS -> 수출 규모 순 |
| 4 | **3단계 월간 갱신** (PART 8-5) | 최근 24개월 안에 수출이 있었던 조합만, 최근 구간 재수집(수정치 덮어쓰기) |
| 5 | **이어받기 / 호출 한도** (PART 8-1) | (시도, HS, 구간)별 상태를 `korea_regional_collect_status` 에 기록. 일일 한도·트래픽 초과 응답 시 멈추고 다음 실행 때 이어서 |
| 6 | **품질 등급** (PART 8-6) | 수집 범위 안 빈 달을 0으로 채워 A(월 예측)/B(분기 예측)/C(부적합)/X(최근 0)/P(수집 미완) -> `korea_regional_series_quality` |
| 7 | **선택 예측** (PART 8-7) | A/B 등급 + 지역 매핑된 계열만 예측 (FORECAST_LINKED_ONLY) |

실행은 PART 8-0 의 스위치를 하나씩 켜고 PART 8-8 을 돌립니다. PART 7 (기업 분석) 은 `RUN_COMPANY_ANALYSIS` 로 켜고 끕니다.


## PART 1. 환경 · Control Panel

In [1]:
# =============================================================================
# PART 1-1. 환경 설정 (노트북 / 데스크탑 공용 경로)
# =============================================================================
import os
import re
import sys
import gc
import time
import math
import logging
import warnings
from pathlib import Path
from datetime import datetime
from typing import Optional, List, Dict, Tuple, Union

import numpy as np
import pandas as pd
import requests
import xmltodict
from requests.adapters import HTTPAdapter
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")

# pandas 버전별 월말/분기말 alias (2.2+: 'ME'/'QE', 이전: 'M'/'Q')
try:
    pd.date_range("2020-01-31", periods=1, freq="ME")
    FREQ_M, FREQ_Q = "ME", "QE"
except Exception:
    FREQ_M, FREQ_Q = "M", "Q"
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("regional_trade")

KNOWN_DATA_PATHS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\DATA",          # 데스크탑
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA",  # 노트북
]


def setup_universal_paths() -> str:
    """DATA 폴더 자동 탐색 + 고정 경로 fallback. 프로젝트 루트와 DATA를 모두 sys.path에 추가."""
    current = Path.cwd()
    for parent in [current, *current.parents]:
        data_folder = parent / "DATA"
        if data_folder.exists():
            for p in (str(parent), str(data_folder)):
                if p not in sys.path:
                    sys.path.insert(0, p)
            print(f"[경로] 프로젝트 루트: {parent} | DATA: {data_folder}")
            return str(data_folder)
    for dp in KNOWN_DATA_PATHS:
        if os.path.exists(dp):
            root = str(Path(dp).parent)
            for p in (root, dp):
                if p not in sys.path:
                    sys.path.insert(0, p)
            print(f"[경로] 고정 경로 사용 | DATA: {dp}")
            return dp
    raise FileNotFoundError(f"DATA 폴더를 찾을 수 없습니다. 현재 위치: {current}")


DATA_PATH = setup_universal_paths()

from DATA.stock_invest_function import get_db_host  # noqa: E402

# 예측 모듈 (DATA 폴더의 universal_ts_forecast_function_v3.py) — 공유 모듈은 수정하지 않고 import만 한다
FORECAST_MODULE = "universal_ts_forecast_function_v3"
import importlib  # noqa: E402
_fc_mod = importlib.import_module(FORECAST_MODULE)
forecast_sarima = _fc_mod.forecast_sarima
forecast_ets = _fc_mod.forecast_ets
forecast_theta = _fc_mod.forecast_theta
print(f"[모듈] {FORECAST_MODULE} import 완료")

DB_INFO = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}


def make_engine(db_info: dict):
    url = (f"mysql+pymysql://{db_info['user']}:{db_info['password']}"
           f"@{db_info['host']}:{int(db_info['port'])}/{db_info['database']}?charset=utf8mb4")
    return create_engine(url, pool_pre_ping=True, pool_recycle=1800,
                         connect_args={"connect_timeout": 10, "read_timeout": 120, "write_timeout": 120})


engine = make_engine(DB_INFO)
print("[DB] engine 준비 완료")


[경로] 프로젝트 루트: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy | DATA: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
[모듈] universal_ts_forecast_function_v3 import 완료
[DB] engine 준비 완료


In [2]:
# =============================================================================
# PART 1-2. Control Panel  —  사용자 입력은 여기 한 곳만 수정
# =============================================================================

# ---- (1) 분석 대상: (ticker, 기업명, HS코드, 시도[, 시군구]) ----
#   - 시도: 이름('제주', '제주도', '경기', '충북'...) 또는 코드('50') 모두 허용
#   - HS: 6자리만 지원 (API가 2/4자리 요청에 resultCode 99 반환). 여러 개는 '854239+854231'
#   - 시군구(선택): 이름('구미시')을 넣으면 해당 시군구 단위 시계열로 분석 (API가 시군구를 반환할 때만)
COMPANY_REGION_INPUTS = [
    ("A080220", "제주반도체", "854239+854231", "제주"),   # HS를 '+'로 묶으면 개별 + 합산 분석
    # ("A178920", "PI첨단소재", "392099", "경북", "구미시"),
]

# ---- (2) API ----
SERVICE_KEY = "2o6NG3ixxDgGQ9S4dWUgsMac9WlxfX46%2BJvFRsAlsXQ6xVi6CZewvNJvbHd4S7exkWwt3YWoKSdwvUNb46kSTQ%3D%3D"
API_URL = "https://apis.data.go.kr/1220000/sigunguperprlstperacrs/getSigunguPerPrlstPerAcrs"

# 요청 파라미터 이름 (명세와 다르면 여기만 수정 — PART 2의 probe_api()로 먼저 확인)
API_PARAM_NAMES = {
    "start": "strtYymm",   # 조회 시작 YYYYMM
    "end":   "endYymm",    # 조회 종료 YYYYMM
    "sido":  "sidoCd",     # 시도코드
    "sgg":   "sggCd",      # 시군구코드 (미사용 시 요청에서 제외)
    "hs":    "hsSgn",      # 품목코드
    "imex":  "imexTpcd",   # 수출입구분 (1=수출, 2=수입) — None이면 요청에서 제외
    "page":  "pageNo",
    "rows":  "numOfRows",
}
API_IMEX_CODE      = None     # 수출만 받으려면 "1". 응답에 수출/수입이 같이 오면 None 유지
API_NUM_OF_ROWS    = 1000
API_AMOUNT_UNIT    = 1000     # 응답 금액(expUsdAmt/impUsdAmt/cmtrBlncAmt) 단위 = 천 달러 -> 달러로 환산하는 배수
                              # (probe 검증: 제주 2026.1~8 반도체 합계 471,191 = 뉴스 약 4.7억 달러)
                              # 전국 테이블 korea_monthly_trade_data 는 달러 단위이므로 반드시 맞춰야 함
MAX_WINDOW_MONTHS  = 12       # 1회 요청 최대 조회기간(개월)
RETRIES, TIMEOUT   = 3, 60
REQUEST_DELAY      = 0.15

# ---- (3) 수집 기간 ----
START_YYYYMM        = "201701"                             # 최초 수집 시작월
END_YYYYMM          = pd.Timestamp.today().strftime("%Y%m") # 최신월 (미발표 월은 빈 응답)
INCREMENTAL         = True    # True: DB 최신월 - REVISION_MONTHS 부터만 재수집
REVISION_MONTHS     = 3       # 최근 N개월은 확정치 수정 반영을 위해 재수집(덮어쓰기)
STORE_SGG_LEVEL     = True    # 시군구 단위 행도 DB에 저장

# ---- (4) 예측 ----
FORECAST_HORIZON_M  = 24      # 월 예측 기간
MIN_MONTHS          = 60      # 월 예측 최소 길이 (universal 모듈 기준과 동일)
MIN_QUARTERS        = 20      # 분기 fallback 최소 길이
MAX_ZERO_SHARE      = 0.20    # 0(무실적) 월 비중이 이보다 크면 월 예측 대신 분기 예측 시도
ENSEMBLE_AGG        = "median"
FORECAST_MODELS     = ["SARIMA", "ETS", "Theta"]

# ---- (5) 기업 매출 연동 ----
IMPLIED_HORIZON_Q   = 4       # 수출함의 매출 산출 분기 수 (기본 4분기)
LAGS                = [0, 1, 2]
MIN_YOY_OBS         = 16
OOS_START           = "2022Q1"
REV_VALUE_ABS_CAP   = 1e14    # 매출예측 SARIMA 발산 필터
RATIO_CV_WARN       = 0.30    # 매출/수출 비율 변동계수 경고 기준
FX_ADJUST           = True    # USD 수출을 원화로 환산 (FinanceDataReader USD/KRW)

# ---- (6) DB 테이블 ----
T_REGION_TRADE    = "korea_regional_trade_data"
T_REGION_FORECAST = "korea_regional_trade_forecast"
T_COMPANY_MAP     = "korea_company_region_hscode_map"
T_NATIONAL_TRADE  = "korea_monthly_trade_data"
T_FS              = "korea_fs_data_from_DG"
T_REV_FORECAST    = "korea_revenue_forecast_result"
DRY_RUN_DB        = False     # True면 DB 쓰기 없이 화면 출력만

# ---- (7) 결과 저장 폴더 ----
def _resolve_out_dir() -> Path:
    for base in [Path(r"C:\Users\82108\OneDrive\INVESTMENT"),
                 Path(r"C:\Users\Hoyoung_Park\OneDrive\INVESTMENT"),
                 Path.home() / "OneDrive" / "INVESTMENT"]:
        if base.exists():
            return base / "한국_지역별수출분석" / datetime.today().strftime("%Y%m")
    return Path.home() / "reports" / "regional_export"

OUT_DIR = Path(os.environ.get("REGION_EXPORT_OUT_DIR", str(_resolve_out_dir())))
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[설정] 대상 {len(COMPANY_REGION_INPUTS)}건 | 수집 {START_YYYYMM}~{END_YYYYMM} | 결과 폴더 {OUT_DIR}")


[설정] 대상 1건 | 수집 201701~202610 | 결과 폴더 C:\Users\82108\OneDrive\INVESTMENT\한국_지역별수출분석\202610


## PART 2. 시도코드 · 관세청 API

In [3]:
# =============================================================================
# PART 2-1. 시도코드 사전 (관세청조회코드_v1.3 '시도코드' 시트 기준)
# =============================================================================
SIDO_CODES = {
    "11": "서울특별시", "12": "전남광주통합특별시", "26": "부산광역시", "27": "대구광역시",
    "28": "인천광역시", "29": "광주광역시", "30": "대전광역시", "31": "울산광역시",
    "36": "세종특별자치시", "41": "경기도", "43": "충청북도", "44": "충청남도",
    "46": "전라남도", "47": "경상북도", "48": "경상남도", "50": "제주특별자치도",
    "51": "강원특별자치도", "52": "전북특별자치도",
}

# 약칭 -> 코드
SIDO_ALIASES = {
    "서울": "11", "부산": "26", "대구": "27", "인천": "28", "광주": "29", "대전": "30", "울산": "31",
    "세종": "36", "경기": "41", "충북": "43", "충청북": "43", "충남": "44", "충청남": "44",
    "전남": "46", "전라남": "46", "경북": "47", "경상북": "47", "경남": "48", "경상남": "48",
    "제주": "50", "제주도": "50", "강원": "51", "강원도": "51", "전북": "52", "전라북": "52",
    "전남광주": "12", "전남광주통합": "12",
}

# 행정구역 통합: 2026.7.1.부터 광주(29)+전남(46) -> 전남광주통합특별시(12)
#   '12'로 분석하면 2026-06까지는 29+46 합산, 2026-07부터는 12를 이어 붙여 연속 시계열을 만든다.
REGION_COMPOSITES = {
    "12": {"before_codes": ["29", "46"], "switch_yymm": "202607"},
}
DISCONTINUED_CODES = {"29": "202607", "46": "202607"}   # 이 월부터 코드 사용 중단


def resolve_sido(x) -> str:
    """'제주', '제주도', '제주특별자치도', 50, '50' -> '50'"""
    s = str(x).strip()
    if s.isdigit():
        s = s.zfill(2)
        if s in SIDO_CODES:
            return s
        raise ValueError(f"알 수 없는 시도코드: {x}")
    for code, name in SIDO_CODES.items():
        if s == name:
            return code
    if s in SIDO_ALIASES:
        return SIDO_ALIASES[s]
    for alias in sorted(SIDO_ALIASES, key=len, reverse=True):
        if s.startswith(alias):
            return SIDO_ALIASES[alias]
    raise ValueError(f"시도명을 해석할 수 없습니다: {x}")


def normalize_hs(x) -> Optional[str]:
    """HS 코드 문자열 정규화 (숫자형 저장으로 소실된 선행 0 복원). 2/4/6/8/10자리 허용."""
    if x is None:
        return None
    s = str(x).strip()
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    s = re.sub(r"\D", "", s)
    if not s:
        return None
    if len(s) % 2 == 1:
        s = "0" + s
    return s if len(s) in (2, 4, 6, 8, 10) else None


def normalize_ticker_a(t) -> Optional[str]:
    s = str(t).strip().upper()
    if len(s) == 7 and s.startswith("A"):
        s = s[1:]
    if s.isdigit():
        s = s.zfill(6)
    return "A" + s if re.fullmatch(r"[0-9A-Z]{6}", s) else None


# =============================================================================
# PART 2-2. 관세청 API 호출 / 파싱
# =============================================================================
def norm_yymm(yymm) -> str:
    s = str(yymm).strip()
    if not re.fullmatch(r"\d{6}", s) or not (1 <= int(s[4:]) <= 12):
        raise ValueError(f"YYYYMM 형식 오류: {yymm}")
    return s


def yymm_to_int(y: str) -> int:
    y = norm_yymm(y)
    return int(y[:4]) * 12 + int(y[4:]) - 1


def int_to_yymm(m: int) -> str:
    return f"{m // 12:04d}{m % 12 + 1:02d}"


def split_windows(start: str, end: str, max_months: int = MAX_WINDOW_MONTHS) -> List[Tuple[str, str]]:
    s, e = yymm_to_int(start), yymm_to_int(end)
    if e < s:
        return []
    out, cur = [], s
    while cur <= e:
        we = min(cur + max_months - 1, e)
        out.append((int_to_yymm(cur), int_to_yymm(we)))
        cur = we + 1
    return out


def make_session(pool: int = 4) -> requests.Session:
    sess = requests.Session()
    ad = HTTPAdapter(pool_connections=pool, pool_maxsize=pool)
    sess.mount("https://", ad)
    sess.mount("http://", ad)
    return sess


def _parse_response(xml_text: str) -> Tuple[pd.DataFrame, Dict]:
    """XML -> (items DataFrame, meta{resultCode, resultMsg, totalCount})"""
    meta = {"resultCode": None, "resultMsg": None, "totalCount": None}
    try:
        parsed = xmltodict.parse(xml_text)
    except Exception:
        meta["resultMsg"] = f"XML 파싱 실패: {xml_text[:200]}"
        return pd.DataFrame(), meta
    root = parsed.get("response") if isinstance(parsed, dict) else None
    if not isinstance(root, dict):
        # 공공데이터포털 인증 오류는 OpenAPI_ServiceResponse 형태로 온다
        err = parsed.get("OpenAPI_ServiceResponse", {}) if isinstance(parsed, dict) else {}
        hdr = err.get("cmmMsgHeader", {}) if isinstance(err, dict) else {}
        meta["resultCode"] = hdr.get("returnReasonCode")
        meta["resultMsg"] = hdr.get("returnAuthMsg") or hdr.get("errMsg") or str(parsed)[:200]
        return pd.DataFrame(), meta
    hdr = root.get("header") or {}
    if isinstance(hdr, dict):
        meta["resultCode"], meta["resultMsg"] = hdr.get("resultCode"), hdr.get("resultMsg")
    body = root.get("body") or {}
    if not isinstance(body, dict):
        return pd.DataFrame(), meta
    meta["totalCount"] = body.get("totalCount")
    items = body.get("items")
    item = items.get("item") if isinstance(items, dict) else None
    if item is None:
        return pd.DataFrame(), meta
    if isinstance(item, dict):
        item = [item]
    return pd.DataFrame(item), meta


def build_url(start: str, end: str, sido_cd: str, hs: str,
              sgg_cd: Optional[str] = None, page: int = 1) -> str:
    """serviceKey는 포털이 준 Encoding 키를 그대로 붙인다(requests params 사용 시 이중 인코딩 위험)."""
    p = API_PARAM_NAMES
    parts = [f"serviceKey={SERVICE_KEY}",
             f"{p['start']}={start}", f"{p['end']}={end}",
             f"{p['sido']}={sido_cd}", f"{p['hs']}={hs}"]
    if sgg_cd and p.get("sgg"):
        parts.append(f"{p['sgg']}={sgg_cd}")
    if API_IMEX_CODE and p.get("imex"):
        parts.append(f"{p['imex']}={API_IMEX_CODE}")
    if p.get("page"):
        parts.append(f"{p['page']}={page}")
    if p.get("rows"):
        parts.append(f"{p['rows']}={API_NUM_OF_ROWS}")
    return API_URL + "?" + "&".join(parts)


def fetch_window(session: requests.Session, start: str, end: str, sido_cd: str, hs: str,
                 sgg_cd: Optional[str] = None) -> Tuple[pd.DataFrame, Dict]:
    """기간 제한 없는 수집 진입점.
    API는 조회기간이 12개월을 넘으면 데이터를 주지 않으므로, 요청 기간을 MAX_WINDOW_MONTHS 이하
    구간으로 자동 분할해 순차 호출한 뒤 합친다. (probe_api, 스캔 셀 등 어디서 호출해도 안전)
    반환 meta: 첫 비정상 구간의 resultCode/resultMsg(모두 정상이면 '00'), 구간별 로그 windows
    """
    start, end = norm_yymm(start), norm_yymm(end)
    wins = split_windows(start, end)
    if not wins:
        raise ValueError(f"종료월이 시작월보다 빠릅니다: {start} ~ {end}")
    frames, logs = [], []
    for ws, we in wins:
        df, m = _fetch_single_window(session, ws, we, sido_cd, hs, sgg_cd)
        logs.append({"win": f"{ws}-{we}", "rows": len(df), "resultCode": m.get("resultCode"),
                     "resultMsg": m.get("resultMsg"), "totalCount": m.get("totalCount")})
        if not df.empty:
            frames.append(df)
    bad = [l for l in logs if str(l["resultCode"]) not in ("00", "0", "None")]
    meta = {"resultCode": bad[0]["resultCode"] if bad else "00",
            "resultMsg": bad[0]["resultMsg"] if bad else "정상서비스",
            "totalCount": sum(int(l["totalCount"] or 0) for l in logs
                              if str(l["totalCount"] or "0").isdigit()),
            "n_windows": len(wins), "windows": logs}
    if bad:
        logger.warning(f"[수집] {sido_cd}/{hs} 비정상 구간 {len(bad)}/{len(wins)}: "
                       f"{[(b['win'], b['resultCode'], b['resultMsg']) for b in bad]}")
    return (pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()), meta


def _fetch_single_window(session: requests.Session, start: str, end: str, sido_cd: str, hs: str,
                         sgg_cd: Optional[str] = None) -> Tuple[pd.DataFrame, Dict]:
    """단일 window(<= 12개월) 수집 (페이지 자동 순회, 재시도). 직접 호출하지 말고 fetch_window 사용."""
    if yymm_to_int(end) - yymm_to_int(start) + 1 > MAX_WINDOW_MONTHS:
        raise ValueError(f"단일 요청 기간 초과: {start}~{end} (> {MAX_WINDOW_MONTHS}개월)")
    frames, page, meta = [], 1, {}
    while True:
        url = build_url(start, end, sido_cd, hs, sgg_cd, page)
        df, meta = pd.DataFrame(), {"resultCode": None, "resultMsg": "no response", "totalCount": None}
        for attempt in range(RETRIES):
            try:
                r = session.get(url, timeout=TIMEOUT)
                if r.status_code in (429, 500, 502, 503, 504) and attempt < RETRIES - 1:
                    time.sleep(2 ** attempt)
                    continue
                r.raise_for_status()
                df, meta = _parse_response(r.text)
                break
            except requests.exceptions.RequestException as e:
                meta["resultMsg"] = str(e)
                if attempt < RETRIES - 1:
                    time.sleep(2 ** attempt)
        if REQUEST_DELAY:
            time.sleep(REQUEST_DELAY)
        if not df.empty:
            frames.append(df)
        try:
            total = int(meta.get("totalCount") or 0)
        except (TypeError, ValueError):
            total = 0
        if df.empty or total <= page * API_NUM_OF_ROWS:
            break
        page += 1
    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if not out.empty:
        out["req_sido_cd"], out["req_hs"] = sido_cd, hs
        out["req_start"], out["req_end"] = start, end
    return out, meta


def probe_api(sido: str = "50", hs: str = "854239", start: str = "202501", end: str = "202608",
              show_rows: int = 10) -> pd.DataFrame:
    """[최초 1회 필수] 실제 응답 구조 확인 (HS는 6자리만 지원 — 2/4자리는 resultCode 99) — 필드명, 시군구 행/총계 행, HS 계층 여부를 눈으로 검증."""
    sido_cd = resolve_sido(sido)
    sess = make_session(1)
    df, meta = fetch_window(sess, norm_yymm(start), norm_yymm(end), sido_cd, hs)
    sess.close()
    masked = build_url(start, end, sido_cd, hs).replace(SERVICE_KEY, "***KEY***")
    print(f"[probe] {masked}")
    print(f"[probe] 요청 {start}~{end} -> {meta.get('n_windows')}개 구간으로 분할 호출")
    for w in meta.get("windows", []):
        print(f"   - {w['win']}: resultCode={w['resultCode']} rows={w['rows']} msg={w['resultMsg']}")
    print(f"[probe] 합계 rows={len(df)}")
    if df.empty:
        print("[probe] 응답 행이 없습니다. API_PARAM_NAMES / 인증키 / 기간을 확인하세요.")
        return df
    print("[probe] 응답 필드:", list(df.columns))
    for c in df.columns:
        if df[c].nunique() <= 30:
            print(f"   - {c}: {sorted(df[c].astype(str).unique().tolist())[:30]}")
    display(df.head(show_rows))
    return df


## PART 3. 표준화 · DB · 수집 · 로드

In [4]:
# =============================================================================
# PART 3-1. 응답 표준화 -> 시도/시군구 월별 long 데이터
# =============================================================================
FIELD_CANDIDATES = {
    "period":  ["year", "yymm", "statYymm", "priodTitle", "ym", "baseYymm"],
    "sido_cd": ["sidoCd", "sido_cd"],
    "sido_nm": ["sidoNm", "sidoName", "sido"],
    "sgg_cd":  ["sggCd", "sigunguCd", "sgguCd", "signguCd"],
    "sgg_nm":  ["sggNm", "sigunguNm", "sgguNm", "signguNm", "sigungu"],
    "hs_row":  ["hsCd", "hsCode", "hsSgn", "prlstCd"],
    "hs_nm":   ["statKor", "hsNm", "korePrlstNm", "prlstNm", "hsKorNm"],
    "expDlr":  ["expDlr", "expUsdAmt", "expAmt"],
    "expWgt":  ["expWgt", "expWght"],
    "expCnt":  ["expCnt", "expCascnt", "expCase"],
    "impDlr":  ["impDlr", "impUsdAmt", "impAmt"],
    "impWgt":  ["impWgt", "impWght"],
    "impCnt":  ["impCnt", "impCascnt", "impCase"],
    "balPayments": ["balPayments", "cmtrBlncAmt", "tradeBal"],
}
VALUE_FIELDS = ["expDlr", "expWgt", "expCnt", "impDlr", "impWgt", "impCnt", "balPayments"]
AMOUNT_FIELDS = {"expDlr", "impDlr", "balPayments"}   # API_AMOUNT_UNIT 배수를 적용할 금액 필드
TOTAL_TOKENS = {"", "전체", "합계", "소계", "총계", "계", "nan", "None", "-"}
SIDO_TOTAL_KEY = "00000"   # 시도 합계 행의 sgg_cd


def _pick_col(cols, cands) -> Optional[str]:
    low = {str(c).lower(): c for c in cols}
    for c in cands:
        if c.lower() in low:
            return low[c.lower()]
    return None


def _period_to_date(v) -> pd.Timestamp:
    d = re.sub(r"\D", "", str(v))
    if len(d) >= 6 and 1 <= int(d[4:6]) <= 12:
        return pd.Timestamp(int(d[:4]), int(d[4:6]), 1) + pd.offsets.MonthEnd(0)
    return pd.NaT


def standardize_raw(raw: pd.DataFrame) -> pd.DataFrame:
    """API 원본 -> 표준 컬럼. 필드명이 달라도 FIELD_CANDIDATES로 자동 매칭."""
    if raw.empty:
        return pd.DataFrame()
    col = {k: _pick_col(raw.columns, v) for k, v in FIELD_CANDIDATES.items()}
    if col["period"] is None:
        raise KeyError(f"기간 필드를 찾지 못했습니다. 응답 필드: {list(raw.columns)} "
                       f"-> FIELD_CANDIDATES['period']에 추가하세요.")
    out = pd.DataFrame(index=raw.index)
    out["date"] = raw[col["period"]].map(_period_to_date)
    out["sido_cd"] = (raw[col["sido_cd"]].astype(str).str.strip() if col["sido_cd"]
                      else raw["req_sido_cd"].astype(str))
    out["sido_nm"] = raw[col["sido_nm"]].astype(str).str.strip() if col["sido_nm"] else ""
    sgg_cd = raw[col["sgg_cd"]].astype(str).str.strip() if col["sgg_cd"] else pd.Series("", index=raw.index)
    sgg_nm = raw[col["sgg_nm"]].astype(str).str.strip() if col["sgg_nm"] else pd.Series("", index=raw.index)
    out["sgg_nm"] = sgg_nm.where(~sgg_nm.isin(["nan", "None"]), "")
    out["sgg_cd"] = sgg_cd.where(~sgg_cd.isin(["", "nan", "None"]), out["sgg_nm"])
    out["hs_row"] = (raw[col["hs_row"]].map(lambda x: re.sub(r"\D", "", str(x))) if col["hs_row"]
                     else raw["req_hs"].astype(str))
    out["hs_nm"] = raw[col["hs_nm"]].astype(str) if col["hs_nm"] else ""
    for f in VALUE_FIELDS:
        c = col.get(f)
        out[f] = pd.to_numeric(raw[c].astype(str).str.replace(",", ""), errors="coerce") if c else np.nan
        if f in AMOUNT_FIELDS:
            out[f] = out[f] * API_AMOUNT_UNIT       # 천 달러 -> 달러
    out["req_sido_cd"], out["req_hs"] = raw["req_sido_cd"].astype(str), raw["req_hs"].astype(str)
    out = out.dropna(subset=["date"])       # '총계' 등 기간이 아닌 행 제거
    return out.reset_index(drop=True)


def _select_hs_level(df: pd.DataFrame, req_hs: str) -> Tuple[pd.DataFrame, str]:
    """HS 계층 중복 합산 방지: 요청코드와 같은 행이 있으면 그 행만, 없으면 가장 상위(짧은) 레벨만 사용."""
    if df["hs_row"].eq("").all():
        return df, "no_hs_field"
    if df["hs_row"].eq(req_hs).any():
        return df[df["hs_row"] == req_hs], "exact"
    sub = df[df["hs_row"].str.startswith(req_hs)]
    if sub.empty:
        return df, "unmatched"     # 응답 코드 체계가 다르면 전체 사용 (probe로 확인 필요)
    min_len = sub["hs_row"].str.len().min()
    return sub[sub["hs_row"].str.len() == min_len], f"children_len{min_len}"


def aggregate_region(std: pd.DataFrame, req_hs: str) -> Tuple[pd.DataFrame, Dict]:
    """표준화 데이터 -> long(date, sido_cd, sgg_cd, sgg_nm, sido_nm, hs_code, indicator, value)
    - 시군구 행과 시도 합계 행이 섞여 오면 합계 행 우선, 없으면 시군구 합산
    """
    diag = {"hs_level": None, "sido_total_source": None, "total_vs_sum_maxdiff": None}
    if std.empty:
        return pd.DataFrame(), diag
    frames = []
    for sido_cd, g in std.groupby("req_sido_cd"):
        g, diag["hs_level"] = _select_hs_level(g, req_hs)
        nm, cd = g["sgg_nm"], g["sgg_cd"]
        is_total = ((nm.isin(TOTAL_TOKENS) & (cd.isin(TOTAL_TOKENS | {SIDO_TOTAL_KEY})
                                               | cd.str.fullmatch(r"\d{2}000")))
                    | nm.isin({"합계", "소계", "총계", "전체", "계"}))
        sido_nm = next((n for n in g["sido_nm"] if n and n not in TOTAL_TOKENS), SIDO_CODES.get(sido_cd, ""))
        sgg_part = g[~is_total]
        tot_part = g[is_total]
        vals = [f for f in VALUE_FIELDS if g[f].notna().any()]

        sgg_m = (sgg_part.groupby(["date", "sgg_cd", "sgg_nm"])[vals].sum(min_count=1).reset_index()
                 if not sgg_part.empty else pd.DataFrame())
        if not tot_part.empty:
            sido_m = tot_part.groupby("date")[vals].sum(min_count=1).reset_index()
            diag["sido_total_source"] = "total_rows"
            if not sgg_m.empty and "expDlr" in vals:
                chk = sgg_m.groupby("date")["expDlr"].sum().to_frame("sgg_sum").join(
                    sido_m.set_index("date")["expDlr"].rename("tot"), how="inner")
                rel = (chk["sgg_sum"] - chk["tot"]).abs() / chk["tot"].replace(0, np.nan)
                diag["total_vs_sum_maxdiff"] = float(rel.max()) if len(rel.dropna()) else None
        elif not sgg_m.empty:
            sido_m = sgg_m.groupby("date")[vals].sum(min_count=1).reset_index()
            diag["sido_total_source"] = "sum_of_sgg"
        else:
            continue
        sido_m["sgg_cd"], sido_m["sgg_nm"] = SIDO_TOTAL_KEY, ""
        parts = [sido_m] + ([sgg_m] if (STORE_SGG_LEVEL and not sgg_m.empty) else [])
        m = pd.concat(parts, ignore_index=True)
        m["sido_cd"], m["sido_nm"], m["hs_code"] = sido_cd, sido_nm, req_hs
        frames.append(m.melt(id_vars=["date", "sido_cd", "sido_nm", "sgg_cd", "sgg_nm", "hs_code"],
                             value_vars=vals, var_name="indicator", value_name="value"))
    if not frames:
        return pd.DataFrame(), diag
    long_df = pd.concat(frames, ignore_index=True).dropna(subset=["value"])
    long_df["sgg_cd"] = long_df["sgg_cd"].astype(str).str.slice(0, 40)
    return long_df, diag


# =============================================================================
# PART 3-2. DB 테이블 (없으면 생성) / 업서트
# =============================================================================
DDL = {
    T_REGION_TRADE: f"""
        CREATE TABLE IF NOT EXISTS `{T_REGION_TRADE}` (
            `date` DATE NOT NULL,
            `sido_cd` VARCHAR(4) NOT NULL,
            `sgg_cd` VARCHAR(40) NOT NULL,
            `hs_code` VARCHAR(10) NOT NULL,
            `indicator` VARCHAR(30) NOT NULL,
            `value` DOUBLE NULL,
            `sido_nm` VARCHAR(40) NULL,
            `sgg_nm` VARCHAR(60) NULL,
            `updated_at` DATETIME DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
            PRIMARY KEY (`date`, `sido_cd`, `sgg_cd`, `hs_code`, `indicator`),
            KEY `idx_hs_sido` (`hs_code`, `sido_cd`)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_general_ci""",
    T_REGION_FORECAST: f"""
        CREATE TABLE IF NOT EXISTS `{T_REGION_FORECAST}` (
            `forecast_date` DATE NOT NULL,
            `date` DATE NOT NULL,
            `freq` CHAR(1) NOT NULL,
            `sido_cd` VARCHAR(4) NOT NULL,
            `sgg_cd` VARCHAR(40) NOT NULL,
            `hs_code` VARCHAR(10) NOT NULL,
            `indicator` VARCHAR(30) NOT NULL,
            `value` DOUBLE NULL,
            `last_actual` DATE NULL,
            `n_obs` INT NULL,
            `created_at` DATETIME DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (`forecast_date`, `date`, `freq`, `sido_cd`, `sgg_cd`, `hs_code`, `indicator`),
            KEY `idx_key` (`hs_code`, `sido_cd`, `sgg_cd`)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_general_ci""",
    T_COMPANY_MAP: f"""
        CREATE TABLE IF NOT EXISTS `{T_COMPANY_MAP}` (
            `ticker` VARCHAR(10) NOT NULL,
            `company_name` VARCHAR(100) NOT NULL,
            `hs_code` VARCHAR(10) NOT NULL,
            `sido_cd` VARCHAR(4) NOT NULL,
            `sgg_nm` VARCHAR(60) NOT NULL DEFAULT '',
            `sido_nm` VARCHAR(40) NULL,
            `is_active` TINYINT NOT NULL DEFAULT 1,
            `note` VARCHAR(200) NULL,
            `created_at` DATETIME DEFAULT CURRENT_TIMESTAMP,
            `updated_at` DATETIME DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
            PRIMARY KEY (`ticker`, `hs_code`, `sido_cd`, `sgg_nm`)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_general_ci""",
}


def ensure_tables():
    if DRY_RUN_DB:
        print("[DRY_RUN] 테이블 생성 생략")
        return
    with engine.begin() as conn:
        for name, ddl in DDL.items():
            conn.execute(text(ddl))
    print(f"[DB] 테이블 확인/생성 완료: {list(DDL)}")


def upsert_rows(table: str, df: pd.DataFrame, key_cols: List[str], upd_cols: List[str],
                chunk: int = 2000) -> int:
    """INSERT ... ON DUPLICATE KEY UPDATE (확정치 수정분을 덮어쓴다)."""
    if df is None or df.empty:
        return 0
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.astype(object).where(pd.notnull(df), None)
    cols = key_cols + [c for c in upd_cols if c not in key_cols]
    sql = text(
        f"INSERT INTO `{table}` ({', '.join(f'`{c}`' for c in cols)}) "
        f"VALUES ({', '.join(f':{c}' for c in cols)}) "
        f"ON DUPLICATE KEY UPDATE {', '.join(f'`{c}`=VALUES(`{c}`)' for c in upd_cols)}"
    )
    rows = df[cols].to_dict("records")
    if DRY_RUN_DB:
        print(f"[DRY_RUN] {table}: {len(rows)}행 업서트 예정")
        return 0
    with engine.begin() as conn:
        for i in range(0, len(rows), chunk):
            conn.execute(sql, rows[i:i + chunk])
    return len(rows)


# =============================================================================
# PART 3-3. 수집 파이프라인 (지역 통합 코드 처리 + 증분 수집)
# =============================================================================
def plan_requests(target_sido: str, start: str, end: str) -> List[Tuple[str, str, str]]:
    """분석 대상 시도 -> 실제 API 요청 [(요청 시도코드, 시작, 종료)]"""
    s, e = yymm_to_int(start), yymm_to_int(end)
    plans = []
    if target_sido in REGION_COMPOSITES:
        comp = REGION_COMPOSITES[target_sido]
        sw = yymm_to_int(comp["switch_yymm"])
        if s < sw:
            for c in comp["before_codes"]:
                plans.append((c, start, int_to_yymm(min(e, sw - 1))))
        if e >= sw:
            plans.append((target_sido, int_to_yymm(max(s, sw)), end))
        return plans
    if target_sido in DISCONTINUED_CODES:
        stop = yymm_to_int(DISCONTINUED_CODES[target_sido])
        if e >= stop:
            logger.warning(f"시도코드 {target_sido}는 {DISCONTINUED_CODES[target_sido]}부터 사용 중단 "
                           f"-> 통합코드 '12' 사용을 권장합니다. 중단 이후 구간은 수집하지 않습니다.")
        if s < stop:
            plans.append((target_sido, start, int_to_yymm(min(e, stop - 1))))
        return plans
    return [(target_sido, start, end)]


def db_last_month(sido_codes: List[str], hs: str) -> Optional[pd.Timestamp]:
    q = text(f"SELECT MAX(`date`) AS d FROM `{T_REGION_TRADE}` "
             f"WHERE hs_code = :hs AND sgg_cd = :tk AND indicator = 'expDlr' AND sido_cd IN :codes")
    from sqlalchemy import bindparam
    q = q.bindparams(bindparam("codes", expanding=True))
    try:
        with engine.connect() as conn:
            d = pd.read_sql(q, conn, params={"hs": hs, "tk": SIDO_TOTAL_KEY, "codes": sido_codes})["d"].iloc[0]
        return pd.Timestamp(d) if pd.notna(d) else None
    except Exception:
        return None


def collect_region_hs(target_sido: str, hs: str, start: str = START_YYYYMM,
                      end: str = END_YYYYMM, incremental: bool = INCREMENTAL) -> Dict:
    """(시도, HS) 1건 수집 -> DB 업서트. 반환: 요약 dict"""
    hs = normalize_hs(hs)
    all_codes = ([target_sido] + REGION_COMPOSITES.get(target_sido, {}).get("before_codes", []))
    if incremental:
        last = db_last_month(all_codes, hs)
        if last is not None:
            inc_start = (last - pd.DateOffset(months=REVISION_MONTHS - 1)).strftime("%Y%m")
            start = max(start, inc_start)
    raws, metas = [], []
    sess = make_session(2)
    try:
        for req_cd, s, e in plan_requests(target_sido, start, end):
            df, meta = fetch_window(sess, s, e, req_cd, hs)      # 12개월 초과 기간은 내부에서 자동 분할
            for w in meta.get("windows", [meta]):
                metas.append({"sido": req_cd, **w})
            if not df.empty:
                raws.append(df)
    finally:
        sess.close()
    meta_df = pd.DataFrame(metas)
    if not raws:
        logger.warning(f"[수집] {target_sido}/{hs}: 응답 없음 {start}~{end}")
        return {"target": target_sido, "hs": hs, "rows_saved": 0, "meta": meta_df, "diag": {}}
    std = standardize_raw(pd.concat(raws, ignore_index=True))
    long_df, diag = aggregate_region(std, hs)
    long_df["date"] = pd.to_datetime(long_df["date"]).dt.date
    n = upsert_rows(T_REGION_TRADE, long_df,
                    key_cols=["date", "sido_cd", "sgg_cd", "hs_code", "indicator"],
                    upd_cols=["value", "sido_nm", "sgg_nm"])
    logger.info(f"[수집] {target_sido}/{hs} {start}~{end}: 원본 {len(std)}행 -> 저장 {n}행 | {diag}")
    return {"target": target_sido, "hs": hs, "rows_saved": n, "meta": meta_df, "diag": diag,
            "long": long_df}


# =============================================================================
# PART 3-4. 시계열 로드 (통합 코드 연결 + 무실적 월 0 채움)
# =============================================================================
def _published_last_month(sido_codes: List[str]) -> Optional[pd.Timestamp]:
    """해당 시도의 전체 HS 기준 최신 적재월 (그 달까지는 행이 없으면 0으로 간주)."""
    from sqlalchemy import bindparam
    q = text(f"SELECT MAX(`date`) AS d FROM `{T_REGION_TRADE}` WHERE sido_cd IN :codes") \
        .bindparams(bindparam("codes", expanding=True))
    with engine.connect() as conn:
        d = pd.read_sql(q, conn, params={"codes": sido_codes})["d"].iloc[0]
    return pd.Timestamp(d) + pd.offsets.MonthEnd(0) if pd.notna(d) else None


def load_region_series(target_sido: str, hs: str, sgg_nm: str = "",
                       indicator: str = "expDlr") -> pd.Series:
    """월말 인덱스 Series. sgg_nm=''이면 시도 합계, 값이 있으면 해당 시군구."""
    from sqlalchemy import bindparam
    hs = normalize_hs(hs)
    comp = REGION_COMPOSITES.get(target_sido)
    codes = [target_sido] + (comp["before_codes"] if comp else [])
    cond = "sgg_cd = :tk" if not sgg_nm else "sgg_nm = :sn"
    q = text(f"SELECT `date`, sido_cd, value FROM `{T_REGION_TRADE}` "
             f"WHERE hs_code = :hs AND indicator = :ind AND {cond} AND sido_cd IN :codes") \
        .bindparams(bindparam("codes", expanding=True))
    with engine.connect() as conn:
        df = pd.read_sql(q, conn, params={"hs": hs, "ind": indicator, "tk": SIDO_TOTAL_KEY,
                                          "sn": sgg_nm, "codes": codes})
    if df.empty:
        return pd.Series(dtype=float, name=f"{target_sido}_{hs}")
    df["date"] = pd.to_datetime(df["date"]) + pd.offsets.MonthEnd(0)
    if comp:
        sw = pd.Timestamp(comp["switch_yymm"][:4] + "-" + comp["switch_yymm"][4:] + "-01")
        df = df[((df["sido_cd"].isin(comp["before_codes"])) & (df["date"] < sw))
                | ((df["sido_cd"] == target_sido) & (df["date"] >= sw))]
    s = df.groupby("date")["value"].sum().sort_index()
    last_pub = _published_last_month(codes)
    end = max(s.index.max(), last_pub) if last_pub is not None else s.index.max()
    full = pd.date_range(s.index.min(), end, freq=FREQ_M)
    s = s.reindex(full).fillna(0.0)      # 적재 범위 내 무행 = 무실적(0)
    s.index.name = "date"
    s.name = f"{target_sido}_{sgg_nm or 'ALL'}_{hs}"
    return s


def load_national_series(hs: str, indicator: str = "expDlr") -> pd.Series:
    q = text(f"SELECT `date`, value FROM `{T_NATIONAL_TRADE}` WHERE root_hs_code = :hs AND indicator = :ind")
    with engine.connect() as conn:
        df = pd.read_sql(q, conn, params={"hs": normalize_hs(hs), "ind": indicator})
    if df.empty:
        return pd.Series(dtype=float)
    df["date"] = pd.to_datetime(df["date"]) + pd.offsets.MonthEnd(0)
    return df.groupby("date")["value"].sum().sort_index()


## PART 4. 지역 수출 예측

In [5]:
# =============================================================================
# PART 4. 지역 수출 예측 (SARIMA / ETS / Theta + median 앙상블)
# =============================================================================
def _run_models(y: pd.Series, horizon: int, m: int) -> Dict[str, np.ndarray]:
    out = {}
    for name in FORECAST_MODELS:
        try:
            if name == "SARIMA":
                r = forecast_sarima(y, horizon, seasonal_period=m)
            elif name == "ETS":
                r = forecast_ets(y, horizon, m=m)
            elif name == "Theta":
                r = forecast_theta(y, horizon, m=m)
            else:
                continue
        except Exception as e:
            r = {"error": str(e)}
        if "forecast" in r:
            fc = np.asarray(r["forecast"], dtype=float)
            if len(fc) == horizon and np.all(np.isfinite(fc)):
                out[name] = fc
    gc.collect()
    return out


def forecast_region_series(s: pd.Series, horizon_m: int = FORECAST_HORIZON_M) -> Dict:
    """월 시계열 예측. 무실적(0) 월이 많으면 분기 합산 시계열로 대체 예측."""
    s = s.dropna().astype(float)
    res = {"freq": None, "n_obs": len(s), "zero_share": None, "fc": None, "status": None}
    if s.empty:
        res["status"] = "no_data"
        return res
    zero_m = float((s <= 0).mean())
    res["zero_share"] = zero_m
    if len(s) >= MIN_MONTHS and zero_m <= MAX_ZERO_SHARE:
        freq, y, h, m = "M", s, horizon_m, 12
    else:
        q = s.resample(FREQ_Q).sum()
        # 마지막 분기가 미완성(월 3개 미만)이면 제외
        if s.index.max() != s.index.max() + pd.offsets.QuarterEnd(0):
            q = q.iloc[:-1]
        zero_q = float((q <= 0).mean()) if len(q) else 1.0
        if len(q) >= MIN_QUARTERS and zero_q <= MAX_ZERO_SHARE:
            freq, y, h, m = "Q", q, int(math.ceil(horizon_m / 3)), 4
            res["zero_share"] = zero_q
        else:
            res["status"] = f"skip_sparse(n={len(s)}, zero_m={zero_m:.0%})"
            return res
    fcs = _run_models(y, h, m)
    if not fcs:
        res["status"] = "all_models_failed"
        return res
    arr = np.vstack(list(fcs.values()))
    ens = np.median(arr, axis=0) if ENSEMBLE_AGG == "median" else arr.mean(axis=0)
    off = pd.offsets.MonthEnd(1) if freq == "M" else pd.offsets.QuarterEnd(1)
    idx = pd.date_range(y.index.max() + off, periods=h, freq=FREQ_M if freq == "M" else FREQ_Q)
    fc = pd.DataFrame(fcs, index=idx)
    fc["ensemble"] = ens
    recent_max = float(y.tail(24 if freq == "M" else 8).max())
    res.update({"freq": freq, "n_obs": len(y), "fc": fc, "last_actual": y.index.max(),
                "status": "ok" if ens.max() <= recent_max * 5 else "ok_but_blowout_flag"})
    return res


def forecast_and_save(target_sido: str, hs: str, sgg_nm: str = "",
                      forecast_date: Optional[str] = None) -> Dict:
    s = load_region_series(target_sido, hs, sgg_nm)
    r = forecast_region_series(s)
    r["series"] = s
    if r["fc"] is None:
        logger.warning(f"[예측] {target_sido}/{sgg_nm or 'ALL'}/{hs}: {r['status']}")
        return r
    fd = forecast_date or datetime.today().strftime("%Y-%m-%d")
    fc = r["fc"].rename(columns={c: (f"{c}_expDlr" if c != "ensemble" else "ensemble_expDlr")
                                 for c in r["fc"].columns})
    long_fc = fc.reset_index().melt(id_vars="index", var_name="indicator", value_name="value") \
        .rename(columns={"index": "date"})
    long_fc["date"] = pd.to_datetime(long_fc["date"]).dt.date
    long_fc["forecast_date"] = fd
    long_fc["freq"] = r["freq"]
    long_fc["sido_cd"] = target_sido
    long_fc["sgg_cd"] = SIDO_TOTAL_KEY if not sgg_nm else sgg_nm[:40]
    long_fc["hs_code"] = normalize_hs(hs)
    long_fc["last_actual"] = pd.Timestamp(r["last_actual"]).date()
    long_fc["n_obs"] = int(r["n_obs"])
    upsert_rows(T_REGION_FORECAST, long_fc,
                key_cols=["forecast_date", "date", "freq", "sido_cd", "sgg_cd", "hs_code", "indicator"],
                upd_cols=["value", "last_actual", "n_obs"])
    logger.info(f"[예측] {target_sido}/{sgg_nm or 'ALL'}/{hs}: freq={r['freq']} n={r['n_obs']} "
                f"models={list(r['fc'].columns)} status={r['status']}")
    return r


## PART 5~6. 기업 매핑 · 매출 연동 분석

In [6]:
# =============================================================================
# PART 5. 기업-지역-HS 매핑 테이블 (korea_company_region_hscode_map)
# =============================================================================
def parse_company_inputs(inputs: List[tuple]) -> pd.DataFrame:
    rows = []
    for e in inputs:
        t, n, h, r = e[0], e[1], e[2], e[3]
        sgg = e[4] if len(e) > 4 and e[4] else ""
        tk, sd = normalize_ticker_a(t), resolve_sido(r)
        hs_list = [normalize_hs(x) for x in re.split(r"[+,/\s]+", str(h)) if x.strip()]
        if tk is None or not hs_list or any(x is None or len(x) != 6 for x in hs_list):
            raise ValueError(f"입력 형식 오류(HS는 6자리): {e}")
        if sd in DISCONTINUED_CODES:
            logger.warning(f"{n}: 시도 {sd}는 {DISCONTINUED_CODES[sd]}부터 코드 중단 -> '12'(전남광주통합) 권장")
        for hs in hs_list:
            rows.append({"ticker": tk, "company_name": str(n).strip(), "hs_code": hs, "sido_cd": sd,
                         "sgg_nm": str(sgg).strip(), "sido_nm": SIDO_CODES[sd], "is_active": 1})
    return pd.DataFrame(rows).drop_duplicates(["ticker", "hs_code", "sido_cd", "sgg_nm"])


def upsert_company_map(inputs: List[tuple]) -> pd.DataFrame:
    df = parse_company_inputs(inputs)
    upsert_rows(T_COMPANY_MAP, df, key_cols=["ticker", "hs_code", "sido_cd", "sgg_nm"],
                upd_cols=["company_name", "sido_nm", "is_active"])
    print(f"[매핑] {len(df)}건 업서트 -> {T_COMPANY_MAP}")
    return df


def load_company_map(active_only: bool = True) -> pd.DataFrame:
    q = f"SELECT * FROM `{T_COMPANY_MAP}`" + (" WHERE is_active = 1" if active_only else "")
    with engine.connect() as conn:
        return pd.read_sql(text(q), conn)


# =============================================================================
# PART 6-1. 매출 실적 / 매출 시계열 예측 로드
# =============================================================================
def load_revenue_actual(ticker: str) -> pd.Series:
    """DataGuide 분기 매출(천원). item_code OR indicator 이중 매칭, 중복 분기는 item_code 행 우선."""
    q = text(f"""
        SELECT `date`, value, CASE WHEN TRIM(item_code) = 'M000904001' THEN 0 ELSE 1 END AS pri
        FROM `{T_FS}`
        WHERE ticker = :t AND (TRIM(item_code) = 'M000904001' OR TRIM(indicator) IN ('매출액', '매출액(천원)'))
    """)
    with engine.connect() as conn:
        df = pd.read_sql(q, conn, params={"t": normalize_ticker_a(ticker)})
    if df.empty:
        return pd.Series(dtype=float)
    df["q"] = pd.to_datetime(df["date"]).dt.to_period("Q")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["value"]).sort_values(["q", "pri"]).drop_duplicates("q", keep="first")
    return df.set_index("q")["value"].sort_index()


def load_revenue_forecast(ticker: str, last_actual_q: Optional[pd.Period] = None) -> pd.Series:
    """최신 created_at 버전, 두 ticker 형식 동시 조회, 발산값 제거 후 SARIMA/ETS/Theta 중앙값."""
    tk = normalize_ticker_a(ticker)
    q = text(f"""
        SELECT `date`, indicator, value, created_at FROM `{T_REV_FORECAST}`
        WHERE ticker IN (:a, :b) AND created_at = (
            SELECT MAX(created_at) FROM `{T_REV_FORECAST}` WHERE ticker IN (:a, :b))
    """)
    with engine.connect() as conn:
        df = pd.read_sql(q, conn, params={"a": tk, "b": tk[1:]})
    if df.empty:
        return pd.Series(dtype=float)
    df = df[df["indicator"].isin(["SARIMA", "ETS", "Theta"])]
    df = df[pd.to_numeric(df["value"], errors="coerce").abs() < REV_VALUE_ABS_CAP]
    df["q"] = pd.to_datetime(df["date"]).dt.to_period("Q")
    s = df.groupby("q")["value"].median().sort_index()
    if last_actual_q is not None:
        s = s[s.index > last_actual_q]       # stale 예측 분기 제거
    s.attrs["created_at"] = str(df["created_at"].max()) if len(df) else None
    return s


def load_region_forecast(target_sido: str, hs: str, sgg_nm: str = "") -> Tuple[pd.DataFrame, Optional[str]]:
    """최신 forecast_date의 지역 예측 (ensemble + 모델별)."""
    q = text(f"""
        SELECT forecast_date, `date`, freq, indicator, value FROM `{T_REGION_FORECAST}`
        WHERE sido_cd = :s AND sgg_cd = :g AND hs_code = :h AND forecast_date = (
            SELECT MAX(forecast_date) FROM `{T_REGION_FORECAST}`
            WHERE sido_cd = :s AND sgg_cd = :g AND hs_code = :h)
    """)
    with engine.connect() as conn:
        df = pd.read_sql(q, conn, params={"s": target_sido, "g": SIDO_TOTAL_KEY if not sgg_nm else sgg_nm[:40],
                                          "h": normalize_hs(hs)})
    if df.empty:
        return pd.DataFrame(), None
    df["date"] = pd.to_datetime(df["date"])
    return df, str(df["forecast_date"].iloc[0])


def load_fx_monthly(start: str = "2016-01-01") -> Optional[pd.Series]:
    """USD/KRW 월평균 (FinanceDataReader). 실패 시 None."""
    try:
        import FinanceDataReader as fdr
        fx = fdr.DataReader("USD/KRW", start)["Close"].astype(float)
        fx = fx.resample(FREQ_M).mean().dropna()
        fx.index = fx.index + pd.offsets.MonthEnd(0)
        return fx
    except Exception as e:
        logger.warning(f"[FX] USD/KRW 로드 실패 -> 원화 환산(M1 비율법) 생략: {e}")
        return None


# =============================================================================
# PART 6-2. 지역 수출 분기 시계열 (실적 + 예측, 월 nowcast 포함)
# =============================================================================
def build_export_quarterly(actual_m: pd.Series, fc_df: pd.DataFrame,
                           fx_m: Optional[pd.Series]) -> pd.DataFrame:
    """분기 수출(USD, 선택 시 원화 천원) + 구간 표시.
    - 실적이 있는 달은 실적, 없는 달은 월 예측(ensemble)으로 채워 분기 합산 (nowcast)
    - 분기 예측(freq=Q)만 있으면 실적 완결 분기 이후는 분기 예측값 사용
    """
    last_act = actual_m.index.max()
    parts = actual_m.rename("usd").to_frame()
    parts["src"] = "A"
    fc_freq = None
    if not fc_df.empty:
        ens = fc_df[fc_df["indicator"] == "ensemble_expDlr"].set_index("date")["value"].sort_index()
        fc_freq = fc_df["freq"].iloc[0]
        if fc_freq == "M":
            ens.index = ens.index + pd.offsets.MonthEnd(0)
            f = ens[ens.index > last_act].rename("usd").to_frame()
            f["src"] = "F"
            parts = pd.concat([parts, f])
    if fx_m is not None and FX_ADJUST:
        fx_full = fx_m.reindex(parts.index).ffill().fillna(fx_m.iloc[-1])  # 미래 환율 = 최근값 고정
        parts["krw_th"] = parts["usd"] * fx_full / 1000.0                    # 천원 (DG 매출 단위와 동일)
    parts["q"] = parts.index.to_period("Q")
    agg = {"usd": "sum", "src": lambda x: "A" if (x == "A").all() else ("F" if (x == "F").all() else "A+F")}
    if "krw_th" in parts:
        agg["krw_th"] = "sum"
    qdf = parts.groupby("q").agg(agg)
    qdf["n_months"] = parts.groupby("q").size()
    qdf = qdf[qdf["n_months"] == 3].drop(columns="n_months")
    if fc_freq == "Q" and not fc_df.empty:
        ensq = fc_df[fc_df["indicator"] == "ensemble_expDlr"].copy()
        ensq["q"] = ensq["date"].dt.to_period("Q")
        ensq = ensq.set_index("q")["value"]
        ensq = ensq[ensq.index > (qdf.index.max() if len(qdf) else ensq.index.min() - 1)]
        add = pd.DataFrame({"usd": ensq, "src": "F"})
        if "krw_th" in qdf and fx_m is not None:
            add["krw_th"] = add["usd"] * float(fx_m.iloc[-1]) / 1000.0
        qdf = pd.concat([qdf, add])
    return qdf.sort_index()


# =============================================================================
# PART 6-3. 수출-매출 연동성 검증 + 수출함의 매출
# =============================================================================
def _hac_ols(y: np.ndarray, x: np.ndarray):
    import statsmodels.api as sm
    X = sm.add_constant(x)
    return sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 4})


def linkage_test(rev_q: pd.Series, exp_q: pd.Series) -> pd.DataFrame:
    """분기 YoY 로그성장률: dlog4(Rev_t) ~ dlog4(Exp_{t-L}). 실적 구간만 사용."""
    from scipy import stats
    r = np.log(rev_q.where(rev_q > 0)).diff(4)
    e = np.log(exp_q.where(exp_q > 0)).diff(4)
    rows = []
    for L in LAGS:
        d = pd.concat([r.rename("r"), e.shift(L).rename("e")], axis=1).dropna()
        row = {"lag": L, "n": len(d)}
        if len(d) < MIN_YOY_OBS:
            rows.append({**row, "grade_note": "insufficient_history"})
            continue
        fit = _hac_ols(d["r"].values, d["e"].values)
        pr, pp = stats.pearsonr(d["r"], d["e"])
        sr, _ = stats.spearmanr(d["r"], d["e"])
        # 표본외: OOS_START부터 expanding window, 1분기 앞 예측 vs naive(직전 YoY)
        oos_err, naive_err = [], []
        for t in d.index[d.index >= pd.Period(OOS_START, "Q")]:
            tr = d[d.index < t]
            if len(tr) < MIN_YOY_OBS - 4:
                continue
            f = _hac_ols(tr["r"].values, tr["e"].values)
            pred = f.params[0] + f.params[1] * d.loc[t, "e"]
            oos_err.append(d.loc[t, "r"] - pred)
            naive_err.append(d.loc[t, "r"] - tr["r"].iloc[-1])
        ratio = (np.sqrt(np.mean(np.square(oos_err))) / np.sqrt(np.mean(np.square(naive_err)))
                 if len(oos_err) >= 4 else np.nan)
        rows.append({**row, "alpha": fit.params[0], "beta": fit.params[1], "t_hac": fit.tvalues[1],
                     "p_hac": fit.pvalues[1], "r2": fit.rsquared, "pearson": pr, "spearman": sr,
                     "abs_p_minus_s": abs(pr - sr), "oos_rmse_ratio": ratio, "n_oos": len(oos_err)})
    return pd.DataFrame(rows)


def share_drift(rev_q: pd.Series, exp_q_krw: pd.Series) -> Dict:
    """log(매출/수출) 선형추세 기울기(연율). 점유율 급변 시 수출은 기업 정보를 담지 못한다."""
    from scipy import stats
    d = pd.concat([rev_q.rename("r"), exp_q_krw.rename("e")], axis=1).dropna()
    d = d[(d["r"] > 0) & (d["e"] > 0)]
    if len(d) < 12:
        return {"drift_slope_ann": np.nan, "drift_p": np.nan}
    y = np.log(d["r"] / d["e"]).values
    x = np.arange(len(y)) / 4.0
    sl = stats.linregress(x, y)
    return {"drift_slope_ann": sl.slope, "drift_p": sl.pvalue}


def grade_linkage(best: pd.Series, drift: Dict) -> str:
    if best is None or pd.isna(best.get("beta", np.nan)):
        return "C"
    drift_ok = (pd.isna(drift["drift_p"]) or drift["drift_p"] > 0.05
                or abs(drift["drift_slope_ann"]) <= 0.15)
    a = (best["beta"] > 0 and best["p_hac"] < 0.05 and (best["oos_rmse_ratio"] < 1)
         and best["abs_p_minus_s"] <= 0.3 and drift_ok)
    if a:
        return "A"
    if best["beta"] > 0 and best["p_hac"] < 0.10:
        return "B"
    return "C"


def implied_revenue(rev_q: pd.Series, exp_q: pd.DataFrame, lk_best: Optional[pd.Series],
                    horizon_q: int = IMPLIED_HORIZON_Q) -> pd.DataFrame:
    """Q1..Qh 수출함의 매출(천원)
    - M1 비율법: 최근 4분기 (매출합 / 원화수출합) x 미래 분기 원화수출
    - M2 탄력성: dlog4 Rev = a + b * dlog4 Exp(t-L), 전년 동분기 매출(없으면 앞서 만든 추정치)에 적용
    """
    q0 = rev_q.index.max()
    fut = pd.period_range(q0 + 1, periods=horizon_q, freq="Q")
    out = pd.DataFrame(index=fut)
    out["export_usd"] = exp_q["usd"].reindex(fut)
    out["export_src"] = exp_q["src"].reindex(fut)
    if "krw_th" in exp_q:
        ttm = pd.period_range(q0 - 3, q0, freq="Q")
        denom = exp_q["krw_th"].reindex(ttm).sum()
        ratio = rev_q.reindex(ttm).sum() / denom if denom > 0 else np.nan
        out["M1_ratio_rev"] = ratio * exp_q["krw_th"].reindex(fut)
        out.attrs["ttm_ratio"] = ratio
    if lk_best is not None and pd.notna(lk_best.get("beta", np.nan)):
        L, a, b = int(lk_best["lag"]), lk_best["alpha"], lk_best["beta"]
        base_exp = exp_q["krw_th"] if "krw_th" in exp_q else exp_q["usd"]
        le = np.log(base_exp.where(base_exp > 0))
        rev_path = rev_q.copy()
        vals = []
        for t in fut:
            g = le.get(t - L, np.nan) - le.get(t - L - 4, np.nan)
            prev = rev_path.get(t - 4, np.nan)
            v = prev * np.exp(a + b * g) if pd.notna(g) and pd.notna(prev) else np.nan
            vals.append(v)
            rev_path.loc[t] = v
        out["M2_elastic_rev"] = vals
    return out


def ratio_stability(rev_q: pd.Series, exp_krw_q: pd.Series, n: int = 8) -> float:
    d = pd.concat([rev_q, exp_krw_q], axis=1).dropna().tail(n)
    if len(d) < 4:
        return np.nan
    rt = d.iloc[:, 0] / d.iloc[:, 1].replace(0, np.nan)
    return float(rt.std() / rt.mean()) if rt.mean() else np.nan


def _forecast_to_frame(r: Dict) -> pd.DataFrame:
    """forecast_region_series() 결과를 load_region_forecast()와 같은 형태로 변환 (DB 미저장 합산용)."""
    if r.get("fc") is None:
        return pd.DataFrame()
    f = r["fc"][["ensemble"]].rename(columns={"ensemble": "value"}).reset_index()
    f = f.rename(columns={f.columns[0]: "date"})
    f["indicator"], f["freq"] = "ensemble_expDlr", r["freq"]
    f["date"] = pd.to_datetime(f["date"])
    return f


def analyze_company(row: pd.Series, fx_m: Optional[pd.Series]) -> Dict:
    """단일 (기업, 지역, HS) 분석. 예측은 DB에 저장된 최신 회차 사용."""
    tk, name, hs, sd, sgg = row["ticker"], row["company_name"], row["hs_code"], row["sido_cd"], row["sgg_nm"]
    act_m = load_region_series(sd, hs, sgg)
    fc_df, fd = (load_region_forecast(sd, hs, sgg) if not act_m.empty else (pd.DataFrame(), None))
    return _analyze_core(tk, name, hs, sd, sgg, act_m, fc_df, fd, fx_m)


def analyze_company_group(tk: str, name: str, sd: str, sgg: str, hs_list: List[str],
                          fx_m: Optional[pd.Series]) -> Dict:
    """같은 기업·지역의 여러 HS를 합산한 시계열로 분석. 합산 시계열을 직접 다시 예측한다."""
    parts = [load_region_series(sd, h, sgg) for h in hs_list]
    parts = [p for p in parts if not p.empty]
    if not parts:
        act_m = pd.Series(dtype=float)
    else:
        act_m = pd.concat(parts, axis=1).fillna(0.0).sum(axis=1)
        act_m.index.name = "date"
    fr = forecast_region_series(act_m) if not act_m.empty else {"fc": None}
    fc_df = _forecast_to_frame(fr)
    fd = datetime.today().strftime("%Y-%m-%d") + " (합산 재예측)" if not fc_df.empty else None
    res = _analyze_core(tk, name, "+".join(hs_list), sd, sgg, act_m, fc_df, fd, fx_m)
    res["combined_fc_status"] = fr.get("status")
    return res


def _analyze_core(tk: str, name: str, hs: str, sd: str, sgg: str, act_m: pd.Series,
                  fc_df: pd.DataFrame, fd: Optional[str], fx_m: Optional[pd.Series]) -> Dict:
    res = {"ticker": tk, "company_name": name, "hs_code": hs, "sido_cd": sd, "sido_nm": SIDO_CODES.get(sd),
           "sgg_nm": sgg}
    if act_m.empty:
        res["status"] = "지역 수출 데이터 없음"
        return res
    exp_q = build_export_quarterly(act_m, fc_df, fx_m)
    rev = load_revenue_actual(tk)
    if rev.empty:
        res["status"] = "매출 실적 없음"
        return res
    q0 = rev.index.max()
    rev_fc = load_revenue_forecast(tk, q0)
    exp_base = exp_q["krw_th"] if "krw_th" in exp_q else exp_q["usd"]
    exp_act_q = exp_base[exp_q["src"] == "A"]

    lk = linkage_test(rev, exp_act_q)
    best = None
    if "t_hac" in lk and lk["t_hac"].notna().any():
        best = lk.loc[lk["t_hac"].abs().idxmax()]
    drift = share_drift(rev, exp_q["krw_th"]) if "krw_th" in exp_q else {"drift_slope_ann": np.nan, "drift_p": np.nan}
    grade = grade_linkage(best, drift)
    imp = implied_revenue(rev, exp_q, best if grade in ("A", "B") else None)
    ttm_rev = rev.reindex(pd.period_range(q0 - 3, q0, freq="Q")).sum()

    def g12(s):
        s = s.reindex(imp.index)
        return s.sum() / ttm_rev - 1 if s.notna().all() and ttm_rev > 0 else np.nan

    ttm_exp = exp_base.reindex(pd.period_range(q0 - 3, q0, freq="Q")).sum()
    fut_exp = exp_base.reindex(imp.index)
    nat_parts = [load_national_series(h) for h in hs.split("+")]
    nat_parts = [p for p in nat_parts if not p.empty]
    nat = pd.concat(nat_parts, axis=1).fillna(0.0).sum(axis=1) if nat_parts else pd.Series(dtype=float)
    reg_share = (act_m.tail(12).sum() / nat.reindex(act_m.tail(12).index).sum()
                 if not nat.empty and nat.reindex(act_m.tail(12).index).sum() > 0 else np.nan)
    res.update({
        "status": "ok", "Q0": str(q0), "region_fc_date": fd,
        "region_fc_freq": fc_df["freq"].iloc[0] if not fc_df.empty else None,
        "last_export_month": str(act_m.index.max().date()),
        "region_share_of_national_12m": reg_share,
        "g_export_12": fut_exp.sum() / ttm_exp - 1 if fut_exp.notna().all() and ttm_exp > 0 else np.nan,
        "nowcast_quarters": int((imp["export_src"] == "A+F").sum() + (imp["export_src"] == "A").sum()),
        "grade": grade,
        "best_lag": None if best is None else int(best["lag"]),
        "beta": None if best is None else best.get("beta"),
        "t_hac": None if best is None else best.get("t_hac"),
        "r2": None if best is None else best.get("r2"),
        "oos_rmse_ratio": None if best is None else best.get("oos_rmse_ratio"),
        "drift_slope_ann": drift["drift_slope_ann"],
        "ratio_cv_8q": ratio_stability(rev, exp_q["krw_th"]) if "krw_th" in exp_q else np.nan,
        "ttm_rev_to_export": imp.attrs.get("ttm_ratio"),
        "g_rev_TS_12": g12(rev_fc),
        "g_rev_M1_ratio_12": g12(imp["M1_ratio_rev"]) if "M1_ratio_rev" in imp else np.nan,
        "g_rev_M2_elastic_12": g12(imp["M2_elastic_rev"]) if "M2_elastic_rev" in imp else np.nan,
        "rev_fc_created_at": rev_fc.attrs.get("created_at"),
    })
    detail = imp.copy()
    detail["rev_TS_forecast"] = rev_fc.reindex(imp.index)
    hist = pd.DataFrame({"revenue_actual": rev.tail(12)})
    res["_detail"] = pd.concat([hist, detail]).assign(ticker=tk, company_name=name)
    res["_linkage"] = lk.assign(ticker=tk, company_name=name, hs_code=hs, sido_cd=sd)
    res["_exp_q"] = exp_q
    res["_rev"] = rev
    return res


def plot_company(res: Dict):
    import matplotlib.pyplot as plt
    if res.get("status") != "ok":
        print(f"[plot] {res.get('company_name')}: {res.get('status')}")
        return
    det, exp_q = res["_detail"], res["_exp_q"]
    fig, ax1 = plt.subplots(figsize=(12, 5))
    x = [p.to_timestamp(how="end") for p in det.index]
    to_eok = 1e5    # 천원 -> 억원
    ax1.plot(x, det["revenue_actual"] / to_eok, "k-o", label="Revenue actual")
    for c, sty in [("rev_TS_forecast", "b--s"), ("M1_ratio_rev", "g--^"), ("M2_elastic_rev", "r--d")]:
        if c in det:
            ax1.plot(x, det[c] / to_eok, sty, label=c)
    ax1.set_ylabel("Revenue (KRW 100M)")
    ax2 = ax1.twinx()
    e = exp_q.reindex(det.index)
    ax2.bar(x, e["usd"] / 1e6, width=40, alpha=0.2, color="gray", label="Regional export (USD mn)")
    ax2.set_ylabel("Regional export (USD mn)")
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=9)
    ax1.set_title(f"{res['ticker']} | HS {res['hs_code']} | sido {res['sido_cd']} {res['sgg_nm']} | "
                  f"grade {res['grade']}")
    ax1.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


## PART 8. [v4] 대량 수집 (3단계) · 품질 등급 · 선택 예측 — 정의

아래 셀들은 함수 정의만 합니다. 실행은 맨 아래 PART 8-8.

In [7]:
# =============================================================================
# PART 8-0. [v4] 대량 수집 Control Panel  —  3단계 수집(지도 -> 과거 채우기 -> 월간 갱신)
# =============================================================================
# [v6] 아래 스위치는 수동 실행용 (PART 8-8). 평소에는 모두 False 로 두고 PART 10 자동 실행을 쓴다
RUN_BUILD_UNIVERSE  = False   # 대상 HS 6자리 목록 만들기 (빠름, API 호출 없음)
RUN_STAGE1_SCAN     = False   # 1단계: 모든 (시도 x HS) 최근 12개월 1구간 -> 수출 지도
RUN_STAGE2_BACKFILL = False   # 2단계: 수출 있는 조합만 START_YYYYMM 까지 과거 채우기
RUN_STAGE3_UPDATE   = False   # 3단계: 월간 갱신 (수출 있는 조합만 최근 구간 재수집)
RUN_QUALITY         = False   # 품질 등급 계산 (API 호출 없음)
RUN_SELECT_FORECAST = False   # 품질 통과 + 기업 연결 계열만 예측

# ---- 호출 한도 ----
DAILY_CALL_LIMIT    = 50000   # 공공데이터포털 이 API 의 일일 트래픽 한도 (마이페이지 > 활용신청 현황에서 확인)
RUN_CALL_LIMIT      = 50000   # 이번 실행에서 쓸 최대 호출 수 (오늘 이미 쓴 호출을 빼고 둘 중 작은 값 적용)
SAFETY_MARGIN       = 200     # 한도 직전 여유분

# ---- 대상 HS 코드 원천 ----
UNIVERSE_FROM_NATIONAL_MAP = True    # DATA/hs_code_map_v2.py 의 HS_CODE_MAP
UNIVERSE_FROM_KR_HS_MAP    = True    # DB kr_company_hs_map
UNIVERSE_FROM_DRIVER_MAP   = True    # DB fd_driver_map_2d (status=driver 의 수출 코드)
UNIVERSE_FROM_COMPANY_MAP  = True    # DB korea_company_region_hscode_map (지역 매핑)
MANUAL_HS_CODES = ["7320", "732010", "732020", "940120", "6704"]   # 수동 후보 (4자리는 6자리로 펼침)
HS6_MASTER_CSV  = r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\reference\hs_codes_master.csv"          # (선택) HS 6자리 전체 목록 CSV 경로. 열 이름에 'hs' 가 들어간 열을 읽음.
                              # 4자리 코드를 6자리로 펼칠 때 전국 테이블에 없는 하위 코드를 보충

# ---- 1단계 (지도) ----
SCAN_WINDOW_END   = "202608"     # None 이면 자동: 오늘 기준 2개월 전 달 (미발표 달 회피)
SCAN_SIDO_CODES   = [c for c in SIDO_CODES if c not in DISCONTINUED_CODES]   # 29/46 대신 12(통합) 사용
SAVE_SCAN_DATA    = True      # 1단계에서 받은 최근 12개월 자료도 korea_regional_trade_data 에 저장

# ---- 2단계 (과거 채우기) ----
STAGE2_MIN_USD_12M = 0        # 최근 12개월 수출이 이 값(달러) 초과인 조합만 과거 채우기
STAGE2_PRIORITY_ONLY = False  # True 면 기업과 연결된 조합(priority 1~2)만

# ---- 3단계 (월간 갱신) ----
STAGE3_ACTIVE_MONTHS = 24     # 최근 N개월 안에 수출이 있었던 조합만 갱신

# ---- 품질 등급 ----
Q_MIN_MONTHS   = 60           # A: 월 예측 최소 길이
Q_MAX_ZERO_M   = 0.20         # A: 0인 달 비율 상한
Q_MIN_QUARTERS = 20           # B: 분기 예측 최소 길이
Q_MAX_ZERO_Q   = 0.20         # B: 0인 분기 비율 상한
Q_MIN_USD_A    = 1_000_000    # A: 연평균 수출(달러) 하한
Q_MIN_USD_B    = 300_000      # B: 연평균 수출(달러) 하한
Q_LEVELS       = ["sido", "sgg"]   # 등급을 매길 단위 (시도 합계, 시군구)

# ---- 선택 예측 ----
FORECAST_GRADES      = ["A", "B"]   # 이 등급만 예측
FORECAST_LINKED_ONLY = True         # True: korea_company_region_hscode_map 에 있는 (시도, HS[, 시군구])만
FORECAST_MAX_SERIES  = 500          # 한 번 실행에서 예측할 최대 계열 수

# ---- v4 테이블 ----
T_HS_UNIVERSE   = "korea_regional_hs_universe"      # 대상 HS 6자리 목록과 출처
T_SCAN          = "korea_regional_trade_scan"       # 1단계 지도: (시도, HS) 최근 12개월 요약
T_COLLECT_STAT  = "korea_regional_collect_status"   # (시도, HS, 구간)별 수집 상태 (이어받기)
T_SERIES_QUAL   = "korea_regional_series_quality"   # 계열 품질 등급
T_KR_HS_MAP     = "kr_company_hs_map"
T_DRIVER_MAP    = "fd_driver_map_2d"

def _auto_scan_end() -> str:
    t = pd.Timestamp.today() - pd.DateOffset(months=2)
    return t.strftime("%Y%m")

SCAN_END   = norm_yymm(SCAN_WINDOW_END) if SCAN_WINDOW_END else _auto_scan_end()
SCAN_START = int_to_yymm(yymm_to_int(SCAN_END) - 11)
print(f"[v4 설정] 1단계 구간 {SCAN_START}~{SCAN_END} | 시도 {len(SCAN_SIDO_CODES)}개 | "
      f"일일 한도 {DAILY_CALL_LIMIT} | 이번 실행 한도 {RUN_CALL_LIMIT}")


[v4 설정] 1단계 구간 202509~202608 | 시도 16개 | 일일 한도 50000 | 이번 실행 한도 50000


In [8]:
# =============================================================================
# PART 8-1. [v4] 테이블 생성 + 호출 예산 (일일 한도, 이어받기)
# =============================================================================
DDL_V4 = {
    T_HS_UNIVERSE: f"""
        CREATE TABLE IF NOT EXISTS `{T_HS_UNIVERSE}` (
            `hs6` VARCHAR(6) NOT NULL,
            `parent_code` VARCHAR(10) NOT NULL DEFAULT '',
            `sources` VARCHAR(200) NULL,
            `priority` TINYINT NOT NULL DEFAULT 3,
            `linked_tickers` VARCHAR(500) NULL,
            `updated_at` DATETIME DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
            PRIMARY KEY (`hs6`)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_general_ci""",
    T_SCAN: f"""
        CREATE TABLE IF NOT EXISTS `{T_SCAN}` (
            `sido_cd` VARCHAR(4) NOT NULL,
            `hs6` VARCHAR(6) NOT NULL,
            `scan_start` CHAR(6) NOT NULL,
            `scan_end` CHAR(6) NOT NULL,
            `exp_12m_usd` DOUBLE NULL,
            `n_months_pos` INT NULL,
            `n_sgg_pos` INT NULL,
            `top_sgg_nm` VARCHAR(60) NULL,
            `top_sgg_share` DOUBLE NULL,
            `status` VARCHAR(10) NOT NULL,
            `scanned_at` DATETIME DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
            PRIMARY KEY (`sido_cd`, `hs6`),
            KEY `idx_hs` (`hs6`)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_general_ci""",
    T_COLLECT_STAT: f"""
        CREATE TABLE IF NOT EXISTS `{T_COLLECT_STAT}` (
            `req_sido_cd` VARCHAR(4) NOT NULL,
            `hs6` VARCHAR(6) NOT NULL,
            `win_start` CHAR(6) NOT NULL,
            `win_end` CHAR(6) NOT NULL,
            `stage` TINYINT NOT NULL,
            `status` VARCHAR(10) NOT NULL,
            `result_code` VARCHAR(20) NULL,
            `result_msg` VARCHAR(200) NULL,
            `n_rows` INT NULL,
            `n_calls` INT NULL,
            `updated_at` DATETIME DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
            PRIMARY KEY (`req_sido_cd`, `hs6`, `win_start`),
            KEY `idx_upd` (`updated_at`)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_general_ci""",
    T_SERIES_QUAL: f"""
        CREATE TABLE IF NOT EXISTS `{T_SERIES_QUAL}` (
            `sido_cd` VARCHAR(4) NOT NULL,
            `sgg_cd` VARCHAR(40) NOT NULL,
            `hs6` VARCHAR(6) NOT NULL,
            `sgg_nm` VARCHAR(60) NULL,
            `grade` CHAR(1) NOT NULL,
            `grade_reason` VARCHAR(200) NULL,
            `n_months` INT NULL,
            `zero_share_m` DOUBLE NULL,
            `zero_share_q` DOUBLE NULL,
            `avg_usd_year` DOUBLE NULL,
            `last12_usd` DOUBLE NULL,
            `first_pos` DATE NULL,
            `last_pos` DATE NULL,
            `flag_start` TINYINT NULL,
            `flag_stop` TINYINT NULL,
            `computed_at` DATETIME DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
            PRIMARY KEY (`sido_cd`, `sgg_cd`, `hs6`),
            KEY `idx_grade` (`grade`)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_general_ci""",
}


def ensure_tables_v4():
    if DRY_RUN_DB:
        print("[DRY_RUN] v4 테이블 생성 생략")
        return
    with engine.begin() as conn:
        for ddl in DDL_V4.values():
            conn.execute(text(ddl))
    print(f"[DB] v4 테이블 확인/생성 완료: {list(DDL_V4)}")


class CallBudget:
    """이번 실행의 호출 예산. 오늘 이미 쓴 호출(T_COLLECT_STAT 기준)을 빼고 계산."""

    def __init__(self):
        used = 0
        try:
            with engine.connect() as conn:
                used = pd.read_sql(text(
                    f"SELECT COALESCE(SUM(n_calls),0) AS n FROM `{T_COLLECT_STAT}` "
                    f"WHERE DATE(updated_at) = CURDATE()"), conn)["n"].iloc[0]
        except Exception:
            used = 0
        self.used_today_before = int(used)
        self.limit = max(0, min(RUN_CALL_LIMIT, DAILY_CALL_LIMIT - self.used_today_before - SAFETY_MARGIN))
        self.used = 0
        self.stopped_reason = None
        print(f"[예산] 오늘 이미 사용 {self.used_today_before} | 이번 실행 가능 {self.limit}")

    def can_spend(self, n: int = 1) -> bool:
        return self.stopped_reason is None and self.used + n <= self.limit

    def spend(self, n: int):
        self.used += n

    def stop(self, reason: str):
        self.stopped_reason = reason
        print(f"[예산] 수집 중단: {reason}")


LIMIT_CODES = {"22", "LIMITED_NUMBER_OF_SERVICE_REQUESTS_EXCEEDS_ERROR"}


def _is_limit_error(meta: Dict) -> bool:
    code = str(meta.get("resultCode") or "")
    msg = str(meta.get("resultMsg") or "")
    return code in LIMIT_CODES or "LIMITED" in msg.upper() or "초과" in msg


def _is_ok(meta: Dict) -> bool:
    return str(meta.get("resultCode")) in ("00", "0")


def load_done_windows(stage: Optional[int] = None) -> set:
    """이미 끝난 (요청시도, HS, 구간시작) 집합. status ok/empty 만 완료로 본다."""
    q = f"SELECT req_sido_cd, hs6, win_start FROM `{T_COLLECT_STAT}` WHERE status IN ('ok','empty')"
    if stage is not None:
        q += f" AND stage = {int(stage)}"
    try:
        with engine.connect() as conn:
            df = pd.read_sql(text(q), conn)
    except Exception:
        return set()
    return set(map(tuple, df[["req_sido_cd", "hs6", "win_start"]].astype(str).values))


def fetch_one_window_tracked(session, budget: CallBudget, req_sido: str, hs6: str,
                             ws: str, we: str, stage: int) -> Tuple[pd.DataFrame, Dict, bool]:
    """단일 구간(<=12개월) 수집 + 상태 기록. 반환 (원본 df, meta, 계속 여부)."""
    if not budget.can_spend(1):
        if budget.stopped_reason is None:
            budget.stop("이번 실행 호출 예산 소진")
        return pd.DataFrame(), {}, False
    df, meta = _fetch_single_window(session, ws, we, req_sido, hs6)
    try:
        total = int(meta.get("totalCount") or 0)
    except (TypeError, ValueError):
        total = 0
    n_calls = max(1, math.ceil(total / API_NUM_OF_ROWS)) if total else 1
    budget.spend(n_calls)
    if _is_limit_error(meta):
        status = "error"
        budget.stop(f"일일 트래픽 한도 응답 ({meta.get('resultCode')} {meta.get('resultMsg')})")
    elif _is_ok(meta):
        status = "ok" if len(df) else "empty"
    elif str(meta.get("resultCode")) == "03" or "NODATA" in str(meta.get("resultMsg")).upper():
        status = "empty"
    else:
        status = "error"
    upsert_rows(T_COLLECT_STAT, pd.DataFrame([{
        "req_sido_cd": req_sido, "hs6": hs6, "win_start": ws, "win_end": we, "stage": stage,
        "status": status, "result_code": str(meta.get("resultCode"))[:20],
        "result_msg": str(meta.get("resultMsg"))[:200], "n_rows": len(df), "n_calls": n_calls}]),
        key_cols=["req_sido_cd", "hs6", "win_start"],
        upd_cols=["win_end", "stage", "status", "result_code", "result_msg", "n_rows", "n_calls"])
    return df, meta, budget.stopped_reason is None


def save_raw_to_trade(raw: pd.DataFrame, hs6: str) -> int:
    """원본 응답 -> 표준화 -> korea_regional_trade_data 업서트 (v3 함수 재사용)."""
    if raw is None or raw.empty:
        return 0
    std = standardize_raw(raw)
    long_df, _ = aggregate_region(std, hs6)
    if long_df.empty:
        return 0
    long_df["date"] = pd.to_datetime(long_df["date"]).dt.date
    return upsert_rows(T_REGION_TRADE, long_df,
                       key_cols=["date", "sido_cd", "sgg_cd", "hs_code", "indicator"],
                       upd_cols=["value", "sido_nm", "sgg_nm"])


In [9]:
# =============================================================================
# PART 8-2. [v4] 대상 HS 6자리 목록 (출처 합치기, 4자리 -> 6자리 펼치기)
#   priority 1: 지역 매핑(korea_company_region_hscode_map)에 있는 HS
#   priority 2: 기업과 연결된 HS (fd_driver_map_2d, kr_company_hs_map, 수동 후보)
#   priority 3: 전국 수출 수집기 목록(HS_CODE_MAP)에만 있는 HS
# =============================================================================
def _read_table_safe(table: str) -> pd.DataFrame:
    try:
        with engine.connect() as conn:
            return pd.read_sql(text(f"SELECT * FROM `{table}`"), conn)
    except Exception as e:
        print(f"[universe] {table} 읽기 실패 (건너뜀): {e}")
        return pd.DataFrame()


def _find_col(df: pd.DataFrame, keys: List[str]) -> Optional[str]:
    for k in keys:
        for c in df.columns:
            if k in str(c).lower():
                return c
    return None


def _codes_from_national_map() -> List[str]:
    try:
        mod = importlib.import_module("DATA.hs_code_map_v2")
    except Exception:
        try:
            mod = importlib.import_module("hs_code_map_v2")
        except Exception as e:
            print(f"[universe] hs_code_map_v2 import 실패 (건너뜀): {e}")
            return []
    obj = getattr(mod, "HS_CODE_MAP", None)
    if obj is None:
        print("[universe] HS_CODE_MAP 없음 (건너뜀)")
        return []
    cands = []
    if isinstance(obj, dict):
        keys = [normalize_hs(k) for k in obj.keys()]
        if sum(k is not None for k in keys) >= 0.8 * max(1, len(keys)):
            cands = keys
        else:   # 키가 코드가 아니면 값에서 찾는다
            for v in obj.values():
                vals = v if isinstance(v, (list, tuple, set)) else [v]
                cands += [normalize_hs(x) for x in vals]
    elif isinstance(obj, (list, tuple, set)):
        for x in obj:
            if isinstance(x, (list, tuple)) and x:
                x = x[0]
            cands.append(normalize_hs(x))
    out = sorted({c for c in cands if c and len(c) >= 4})
    print(f"[universe] HS_CODE_MAP 코드 {len(out)}개 (예: {out[:5]})")
    return out


def _codes_from_kr_hs_map() -> Tuple[List[str], Dict[str, set]]:
    df = _read_table_safe(T_KR_HS_MAP)
    if df.empty:
        return [], {}
    hc = _find_col(df, ["hs"])
    tc = _find_col(df, ["ticker", "stock_code", "entity", "code"])
    if hc is None:
        print(f"[universe] {T_KR_HS_MAP}: HS 열을 찾지 못함. 열: {list(df.columns)}")
        return [], {}
    link = {}
    for _, r in df.iterrows():
        c = normalize_hs(r[hc])
        if c and len(c) >= 4:
            link.setdefault(c, set()).add(str(r[tc]) if tc else "")
    print(f"[universe] {T_KR_HS_MAP}: 코드 {len(link)}개 (HS 열 '{hc}', 기업 열 '{tc}')")
    return sorted(link), link


def _codes_from_driver_map() -> Tuple[List[str], Dict[str, set]]:
    df = _read_table_safe(T_DRIVER_MAP)
    if df.empty or "code" not in df.columns:
        return [], {}
    if "status" in df.columns:
        df = df[df["status"] == "driver"]
    link = {}
    for _, r in df.iterrows():
        raw = str(r["code"])
        if not re.fullmatch(r"\d{4,10}", raw):     # bok:, kosis_idx: 같은 내수 계열 제외
            continue
        c = normalize_hs(raw)
        if c:
            link.setdefault(c, set()).add(str(r.get("entity_id", "")))
    print(f"[universe] {T_DRIVER_MAP}: 수출 코드 {len(link)}개")
    return sorted(link), link


def _codes_from_company_region_map() -> Tuple[List[str], Dict[str, set], pd.DataFrame]:
    df = _read_table_safe(T_COMPANY_MAP)
    if df.empty:
        return [], {}, df
    link = {}
    for _, r in df.iterrows():
        c = normalize_hs(r["hs_code"])
        if c:
            link.setdefault(c, set()).add(str(r["ticker"]))
    print(f"[universe] {T_COMPANY_MAP}: 코드 {len(link)}개, 매핑 {len(df)}행")
    return sorted(link), link, df


def _hs6_reference() -> set:
    """4자리 -> 6자리 펼치기용 6자리 코드 사전: 전국 테이블의 HS 열 + (선택) HS6_MASTER_CSV."""
    ref = set()
    try:
        with engine.connect() as conn:
            cols = pd.read_sql(text(f"SHOW COLUMNS FROM `{T_NATIONAL_TRADE}`"), conn)["Field"].tolist()
            for c in [c for c in cols if "hs" in c.lower()]:
                vals = pd.read_sql(text(f"SELECT DISTINCT `{c}` AS v FROM `{T_NATIONAL_TRADE}`"), conn)["v"]
                for v in vals:
                    s = normalize_hs(v)
                    if s and len(s) >= 6:
                        ref.add(s[:6])
    except Exception as e:
        print(f"[universe] 전국 테이블 HS 열 읽기 실패: {e}")
    if HS6_MASTER_CSV and not os.path.exists(HS6_MASTER_CSV):
        print(f"[universe] 경고: HS6_MASTER_CSV 파일이 없습니다 -> {HS6_MASTER_CSV}")
    if HS6_MASTER_CSV and os.path.exists(HS6_MASTER_CSV):
        m = pd.read_csv(HS6_MASTER_CSV, dtype=str)
        hc = _find_col(m, ["hs"])
        if hc is None:
            print(f"[universe] 경고: HS6_MASTER_CSV 에 'hs' 가 들어간 열이 없습니다. 열: {list(m.columns)}")
        if hc:
            for v in m[hc]:
                s = normalize_hs(v)
                if s and len(s) >= 6:
                    ref.add(s[:6])
    print(f"[universe] 6자리 참조 사전 {len(ref)}개")
    return ref


def build_hs_universe() -> pd.DataFrame:
    sources: Dict[str, set] = {}     # 원 코드(4~10자리) -> 출처들
    linked: Dict[str, set] = {}
    def add(codes, src, link=None):
        for c in codes:
            sources.setdefault(c, set()).add(src)
        for c, tks in (link or {}).items():
            linked.setdefault(c, set()).update(t for t in tks if t)

    region_df = pd.DataFrame()
    if UNIVERSE_FROM_NATIONAL_MAP:
        add(_codes_from_national_map(), "national_map")
    if UNIVERSE_FROM_KR_HS_MAP:
        c, l = _codes_from_kr_hs_map(); add(c, "kr_hs_map", l)
    if UNIVERSE_FROM_DRIVER_MAP:
        c, l = _codes_from_driver_map(); add(c, "driver_map", l)
    if UNIVERSE_FROM_COMPANY_MAP:
        c, l, region_df = _codes_from_company_region_map(); add(c, "region_map", l)
    add([normalize_hs(x) for x in MANUAL_HS_CODES if normalize_hs(x)], "manual")

    ref = _hs6_reference()
    rows, unresolved = [], []
    for code, srcs in sources.items():
        if len(code) >= 6:
            targets = [code[:6]]
        else:   # 4자리 -> 6자리 하위 코드
            targets = sorted(x for x in ref if x.startswith(code))
            if not targets:
                unresolved.append({"code": code, "sources": ",".join(sorted(srcs))})
                continue
        pri = 1 if "region_map" in srcs else (2 if srcs & {"driver_map", "kr_hs_map", "manual"} else 3)
        for t in targets:
            rows.append({"hs6": t, "parent_code": code if len(code) < 6 else "",
                         "sources": ",".join(sorted(srcs)), "priority": pri,
                         "linked_tickers": ",".join(sorted(linked.get(code, set())))[:500]})
    uni = pd.DataFrame(rows)
    if uni.empty:
        print("[universe] 대상 코드가 없습니다. 출처 설정을 확인하세요.")
        return uni
    # 같은 hs6 가 여러 원 코드에서 오면 가장 높은 우선순위(작은 값)와 출처 합집합
    uni = (uni.sort_values("priority")
              .groupby("hs6", as_index=False)
              .agg({"parent_code": "first", "priority": "min",
                    "sources": lambda s: ",".join(sorted(set(",".join(s).split(",")))),
                    "linked_tickers": lambda s: ",".join(sorted({x for v in s for x in v.split(",") if x}))[:500]}))
    upsert_rows(T_HS_UNIVERSE, uni, key_cols=["hs6"],
                upd_cols=["parent_code", "sources", "priority", "linked_tickers"])
    if unresolved:
        ur = pd.DataFrame(unresolved)
        p = OUT_DIR / f"hs_universe_unresolved_{datetime.today():%Y%m%d}.csv"
        ur.to_csv(p, index=False, encoding="utf-8-sig")
        print(f"[universe] 6자리로 펼치지 못한 4자리 코드 {len(ur)}개 -> {p} (HS6_MASTER_CSV 로 보충 가능)")
    n_calls = len(uni) * len(SCAN_SIDO_CODES)
    print(f"[universe] 대상 HS6 {len(uni)}개 (priority 1: {(uni.priority==1).sum()}, "
          f"2: {(uni.priority==2).sum()}, 3: {(uni.priority==3).sum()}) "
          f"-> 1단계 예상 호출 약 {n_calls:,}회 (통합코드 12 는 구간에 따라 최대 3회)")
    return uni


def load_universe() -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(f"SELECT * FROM `{T_HS_UNIVERSE}` ORDER BY priority, hs6"), conn)


In [10]:
# =============================================================================
# PART 8-3. [v4] 1단계: 수출 지도 (모든 시도 x HS, 최근 12개월 1구간)
#   - 결과는 korea_regional_trade_scan 에 (시도, HS)당 1행. 0인 조합도 '0'으로 남겨 다시 요청하지 않는다.
#   - 받은 최근 12개월 원자료는 korea_regional_trade_data 에도 저장 (SAVE_SCAN_DATA)
# =============================================================================
def _scan_done_keys() -> set:
    try:
        with engine.connect() as conn:
            df = pd.read_sql(text(
                f"SELECT sido_cd, hs6 FROM `{T_SCAN}` WHERE status IN ('ok','empty') "
                f"AND scan_start = :s AND scan_end = :e"), conn, params={"s": SCAN_START, "e": SCAN_END})
        return set(map(tuple, df.astype(str).values))
    except Exception:
        return set()


def _summarize_scan(long_df: pd.DataFrame) -> Dict:
    out = {"exp_12m_usd": 0.0, "n_months_pos": 0, "n_sgg_pos": 0, "top_sgg_nm": None, "top_sgg_share": None}
    if long_df is None or long_df.empty:
        return out
    e = long_df[long_df["indicator"] == "expDlr"]
    tot = e[e["sgg_cd"] == SIDO_TOTAL_KEY].groupby("date")["value"].sum()
    out["exp_12m_usd"] = float(tot.sum())
    out["n_months_pos"] = int((tot > 0).sum())
    sgg = e[e["sgg_cd"] != SIDO_TOTAL_KEY].groupby("sgg_nm")["value"].sum()
    sgg = sgg[sgg > 0]
    out["n_sgg_pos"] = int(len(sgg))
    if len(sgg) and out["exp_12m_usd"] > 0:
        out["top_sgg_nm"] = str(sgg.idxmax())[:60]
        out["top_sgg_share"] = float(sgg.max() / out["exp_12m_usd"])
    return out


def run_stage1_scan(universe: pd.DataFrame, budget: CallBudget) -> pd.DataFrame:
    done = _scan_done_keys()
    todo = [(hs, sd) for hs in universe["hs6"] for sd in SCAN_SIDO_CODES if (sd, hs) not in done]
    print(f"[1단계] 대상 {len(universe) * len(SCAN_SIDO_CODES):,}조합 | 완료 {len(done):,} | 남음 {len(todo):,}")
    sess, log, t0 = make_session(2), [], time.time()
    try:
        for i, (hs, target) in enumerate(todo, 1):
            raws, statuses, stopped = [], [], False
            plans = plan_requests(target, SCAN_START, SCAN_END)
            for req_cd, s, e in plans:
                df, meta, go = fetch_one_window_tracked(sess, budget, req_cd, hs, s, e, stage=1)
                if not meta:            # 예산 소진으로 요청 자체를 안 함
                    stopped = True
                    break
                if len(df):
                    statuses.append("ok")
                    raws.append(df)
                elif _is_ok(meta) or str(meta.get("resultCode")) == "03":
                    statuses.append("empty")
                else:
                    statuses.append("error")
                if not go:
                    stopped = True
                    break
            if stopped and (len(statuses) < len(plans) or "error" in statuses):
                break                   # 조합을 끝내지 못했으면 지도에 쓰지 않고 다음 실행에서 이어받기
            raw = pd.concat(raws, ignore_index=True) if raws else pd.DataFrame()
            long_df = pd.DataFrame()
            if len(raw):
                long_df, _ = aggregate_region(standardize_raw(raw), hs)
                if SAVE_SCAN_DATA:
                    save_raw_to_trade(raw, hs)
            summ = _summarize_scan(long_df)
            status = "error" if "error" in statuses else ("ok" if summ["exp_12m_usd"] > 0 else "empty")
            row = {"sido_cd": target, "hs6": hs, "scan_start": SCAN_START, "scan_end": SCAN_END,
                   "status": status, **summ}
            upsert_rows(T_SCAN, pd.DataFrame([row]), key_cols=["sido_cd", "hs6"],
                        upd_cols=["scan_start", "scan_end", "status", "exp_12m_usd", "n_months_pos",
                                  "n_sgg_pos", "top_sgg_nm", "top_sgg_share"])
            log.append(row)
            if i % 200 == 0:
                el = time.time() - t0
                print(f"   ... {i:,}/{len(todo):,} 조합 | 호출 {budget.used:,} | {el/60:.1f}분 경과")
            if stopped:
                break
    finally:
        sess.close()
    lg = pd.DataFrame(log)
    if len(lg):
        print(f"[1단계] 이번 실행 {len(lg):,}조합 기록 | 수출 있음 {(lg.status=='ok').sum():,} | "
              f"0 {(lg.status=='empty').sum():,} | 오류 {(lg.status=='error').sum():,} | 호출 {budget.used:,}")
    return lg


# =============================================================================
# PART 8-4. [v4] 2단계: 과거 채우기 (수출 있는 조합만, START_YYYYMM ~ 1단계 구간 직전)
# =============================================================================
def stage2_candidates(universe: pd.DataFrame) -> pd.DataFrame:
    with engine.connect() as conn:
        scan = pd.read_sql(text(f"SELECT * FROM `{T_SCAN}` WHERE status = 'ok' AND exp_12m_usd > :m"),
                           conn, params={"m": float(STAGE2_MIN_USD_12M)})
    if scan.empty:
        return scan
    scan = scan.merge(universe[["hs6", "priority", "linked_tickers"]], on="hs6", how="left")
    scan["priority"] = scan["priority"].fillna(3).astype(int)
    try:
        cm = load_company_map()
        cm["hs6"] = cm["hs_code"].map(normalize_hs)
        keys = set(zip(cm["sido_cd"].astype(str), cm["hs6"]))
        scan.loc[[(s, h) in keys for s, h in zip(scan["sido_cd"], scan["hs6"])], "priority"] = 0
    except Exception:
        pass
    if STAGE2_PRIORITY_ONLY:
        scan = scan[scan["priority"] <= 2]
    return scan.sort_values(["priority", "exp_12m_usd"], ascending=[True, False]).reset_index(drop=True)


def run_stage2_backfill(universe: pd.DataFrame, budget: CallBudget) -> pd.DataFrame:
    cand = stage2_candidates(universe)
    hist_end = int_to_yymm(yymm_to_int(SCAN_START) - 1)
    done = load_done_windows()
    jobs = []
    for _, r in cand.iterrows():
        for req_cd, s, e in plan_requests(str(r["sido_cd"]), START_YYYYMM, hist_end):
            for ws, we in split_windows(s, e):
                if (req_cd, r["hs6"], ws) not in done:
                    jobs.append((int(r["priority"]), str(r["sido_cd"]), req_cd, r["hs6"], ws, we))
    n_combo = cand[["sido_cd", "hs6"]].drop_duplicates().shape[0] if len(cand) else 0
    print(f"[2단계] 대상 조합 {n_combo:,} | 남은 구간 {len(jobs):,} (구간당 약 1회 호출)")
    sess, log, t0 = make_session(2), [], time.time()
    try:
        for i, (pri, target, req_cd, hs, ws, we) in enumerate(jobs, 1):
            df, meta, go = fetch_one_window_tracked(sess, budget, req_cd, hs, ws, we, stage=2)
            if not meta:
                break
            n = save_raw_to_trade(df, hs) if len(df) else 0
            log.append({"priority": pri, "target": target, "req_sido_cd": req_cd, "hs6": hs,
                        "win": f"{ws}-{we}", "rows_raw": len(df), "rows_saved": n,
                        "resultCode": meta.get("resultCode")})
            if i % 500 == 0:
                print(f"   ... {i:,}/{len(jobs):,} 구간 | 호출 {budget.used:,} | {(time.time()-t0)/60:.1f}분")
            if not go:
                break
    finally:
        sess.close()
    lg = pd.DataFrame(log)
    if len(lg):
        print(f"[2단계] 이번 실행 {len(lg):,}구간 | 자료 있음 {(lg.rows_raw>0).sum():,} | 호출 {budget.used:,} | "
              f"남은 구간 {len(jobs) - len(lg):,}")
    return lg


# =============================================================================
# PART 8-5. [v4] 3단계: 월간 갱신 (최근 STAGE3_ACTIVE_MONTHS 개월 안에 수출이 있었던 조합)
#   - 조합별 DB 최신월 - (REVISION_MONTHS-1) ~ END_YYYYMM 을 다시 받아 덮어쓴다 (확정치 수정 반영)
#   - 같은 날 이미 갱신한 구간은 건너뛴다
# =============================================================================
def _to_target_sido(sd: str) -> str:
    for tgt, comp in REGION_COMPOSITES.items():
        if sd in comp["before_codes"]:
            return tgt
    return sd


def stage3_combos() -> pd.DataFrame:
    since = (pd.Timestamp.today() - pd.DateOffset(months=STAGE3_ACTIVE_MONTHS)).date()
    with engine.connect() as conn:
        df = pd.read_sql(text(
            f"SELECT sido_cd, hs_code, MAX(`date`) AS last_date FROM `{T_REGION_TRADE}` "
            f"WHERE indicator = 'expDlr' AND sgg_cd = :tk AND value > 0 AND `date` >= :since "
            f"GROUP BY sido_cd, hs_code"), conn, params={"tk": SIDO_TOTAL_KEY, "since": since})
    if df.empty:
        return df
    df["target"] = df["sido_cd"].astype(str).map(_to_target_sido)
    df["last_date"] = pd.to_datetime(df["last_date"])
    return df.groupby(["target", "hs_code"], as_index=False)["last_date"].max()


def run_stage3_update(budget: CallBudget) -> pd.DataFrame:
    combos = stage3_combos()
    if combos.empty:
        print("[3단계] 갱신할 조합 없음")
        return combos
    try:
        with engine.connect() as conn:
            today_done = set(map(tuple, pd.read_sql(text(
                f"SELECT req_sido_cd, hs6, win_start FROM `{T_COLLECT_STAT}` "
                f"WHERE stage = 3 AND DATE(updated_at) = CURDATE() AND status IN ('ok','empty')"),
                conn).astype(str).values))
    except Exception:
        today_done = set()
    jobs = []
    for _, r in combos.iterrows():
        start = (r["last_date"] - pd.DateOffset(months=REVISION_MONTHS - 1)).strftime("%Y%m")
        for req_cd, s, e in plan_requests(r["target"], start, END_YYYYMM):
            for ws, we in split_windows(s, e):
                if (req_cd, r["hs_code"], ws) not in today_done:
                    jobs.append((r["target"], req_cd, r["hs_code"], ws, we))
    print(f"[3단계] 갱신 조합 {len(combos):,} | 구간 {len(jobs):,}")
    sess, log = make_session(2), []
    try:
        for target, req_cd, hs, ws, we in jobs:
            df, meta, go = fetch_one_window_tracked(sess, budget, req_cd, hs, ws, we, stage=3)
            if not meta:
                break
            n = save_raw_to_trade(df, hs) if len(df) else 0
            log.append({"target": target, "req_sido_cd": req_cd, "hs6": hs, "win": f"{ws}-{we}",
                        "rows_raw": len(df), "rows_saved": n})
            if not go:
                break
    finally:
        sess.close()
    lg = pd.DataFrame(log)
    print(f"[3단계] 이번 실행 {len(lg):,}구간 | 호출 {budget.used:,}")
    return lg


In [11]:
# =============================================================================
# PART 8-6. [v4] 품질 등급 (API 호출 없음)
#   - 수집 범위(상태 테이블의 완료 구간) 안에서 행이 없는 달 = 0 으로 채운 뒤 계산
#   - 과거 채우기가 끝나지 않은 조합(수집 시작월 > START_YYYYMM)은 'P'(대기)로 둔다
#   등급: A 월 예측 가능 / B 분기 예측 가능 / C 자료는 있으나 예측 부적합 / X 최근 12개월 0 / P 수집 미완
# =============================================================================
def _coverage_by_target() -> pd.DataFrame:
    with engine.connect() as conn:
        st = pd.read_sql(text(
            f"SELECT req_sido_cd, hs6, MIN(win_start) AS c_start, MAX(win_end) AS c_end "
            f"FROM `{T_COLLECT_STAT}` WHERE status IN ('ok','empty') GROUP BY req_sido_cd, hs6"), conn)
    if st.empty:
        return st
    st["target"] = st["req_sido_cd"].astype(str).map(_to_target_sido)
    return st.groupby(["target", "hs6"], as_index=False).agg(c_start=("c_start", "min"), c_end=("c_end", "max"))


def _grade_series(s: pd.Series, c_start: pd.Timestamp, c_end: pd.Timestamp) -> Dict:
    s = s.reindex(pd.date_range(c_start, c_end, freq=FREQ_M)).fillna(0.0)
    pos = s[s > 0]
    out = {"n_months": int(len(s)), "last12_usd": float(s.tail(12).sum()),
           "first_pos": pos.index.min().date() if len(pos) else None,
           "last_pos": pos.index.max().date() if len(pos) else None}
    if not len(pos):
        out.update(grade="X", grade_reason="수출 없음", zero_share_m=1.0, zero_share_q=1.0,
                   avg_usd_year=0.0, flag_start=0, flag_stop=1)
        return out
    eff = s[s.index >= pos.index.min()]                       # 첫 수출 이후만 유효 기간
    q = eff.resample(FREQ_Q).sum()
    if eff.index.max() != eff.index.max() + pd.offsets.QuarterEnd(0):
        q = q.iloc[:-1]
    zm, zq = float((eff <= 0).mean()), float((q <= 0).mean()) if len(q) else 1.0
    avg = float(eff.mean() * 12)
    flag_start = int(pos.index.min() > c_start + pd.DateOffset(months=12))
    flag_stop = int(pos.index.max() < c_end - pd.DateOffset(months=6))
    out.update(zero_share_m=zm, zero_share_q=zq, avg_usd_year=avg, flag_start=flag_start, flag_stop=flag_stop)
    if out["last12_usd"] <= 0:
        out.update(grade="X", grade_reason="최근 12개월 수출 0")
    elif len(eff) >= Q_MIN_MONTHS and zm <= Q_MAX_ZERO_M and avg >= Q_MIN_USD_A:
        out.update(grade="A", grade_reason=f"월 {len(eff)}개, 0비율 {zm:.0%}, 연 {avg/1e6:.1f}백만$")
    elif len(q) >= Q_MIN_QUARTERS and zq <= Q_MAX_ZERO_Q and avg >= Q_MIN_USD_B:
        out.update(grade="B", grade_reason=f"분기 {len(q)}개, 0비율 {zq:.0%}, 연 {avg/1e6:.1f}백만$")
    else:
        why = []
        if len(eff) < Q_MIN_MONTHS: why.append(f"유효 {len(eff)}개월")
        if zm > Q_MAX_ZERO_M: why.append(f"0비율 월 {zm:.0%}/분기 {zq:.0%}")
        if avg < Q_MIN_USD_B: why.append(f"연 {avg/1e6:.2f}백만$")
        out.update(grade="C", grade_reason=", ".join(why)[:200])
    if flag_start:
        out["grade_reason"] = (out["grade_reason"] + " | 중간 시작").strip()[:200]
    if flag_stop:
        out["grade_reason"] = (out["grade_reason"] + " | 최근 중단").strip()[:200]
    return out


def run_quality() -> pd.DataFrame:
    cov = _coverage_by_target()
    if cov.empty:
        print("[품질] 수집 상태 기록이 없습니다.")
        return cov
    start_full = pd.Timestamp(START_YYYYMM[:4] + "-" + START_YYYYMM[4:] + "-01") + pd.offsets.MonthEnd(0)
    rows = []
    for hs, cg in cov.groupby("hs6"):
        with engine.connect() as conn:
            d = pd.read_sql(text(
                f"SELECT `date`, sido_cd, sgg_cd, sgg_nm, value FROM `{T_REGION_TRADE}` "
                f"WHERE hs_code = :hs AND indicator = 'expDlr'"), conn, params={"hs": hs})
        if len(d):
            d["date"] = pd.to_datetime(d["date"]) + pd.offsets.MonthEnd(0)
            d["target"] = d["sido_cd"].astype(str).map(_to_target_sido)
        for _, c in cg.iterrows():
            tgt = c["target"]
            c_start = pd.Timestamp(c["c_start"][:4] + "-" + c["c_start"][4:] + "-01") + pd.offsets.MonthEnd(0)
            c_end = pd.Timestamp(c["c_end"][:4] + "-" + c["c_end"][4:] + "-01") + pd.offsets.MonthEnd(0)
            sub = d[d["target"] == tgt] if len(d) else d
            if sub is None or len(sub) == 0:
                continue                 # 자료가 전혀 없는 조합(1단계 0)은 기록하지 않는다
            series = {("total", SIDO_TOTAL_KEY, ""):
                      sub[sub["sgg_cd"] == SIDO_TOTAL_KEY].groupby("date")["value"].sum()}
            if "sgg" in Q_LEVELS:
                sg = sub[sub["sgg_cd"] != SIDO_TOTAL_KEY]
                for nm, g in sg.groupby("sgg_nm"):
                    series[("sgg", str(nm)[:40], str(nm))] = g.groupby("date")["value"].sum()
            for (lvl, key, nm), s in series.items():
                if lvl == "total" and "sido" not in Q_LEVELS:
                    continue
                g = _grade_series(s, c_start, c_end)
                if c_start > start_full:
                    g["grade"] = "P"
                    g["grade_reason"] = f"수집 미완 (시작 {c['c_start']}) | " + g["grade_reason"]
                rows.append({"sido_cd": tgt, "sgg_cd": key, "hs6": hs, "sgg_nm": nm, **g})
    q = pd.DataFrame(rows)
    if q.empty:
        return q
    q["grade_reason"] = q["grade_reason"].str.slice(0, 200)
    upsert_rows(T_SERIES_QUAL, q, key_cols=["sido_cd", "sgg_cd", "hs6"],
                upd_cols=["sgg_nm", "grade", "grade_reason", "n_months", "zero_share_m", "zero_share_q",
                          "avg_usd_year", "last12_usd", "first_pos", "last_pos", "flag_start", "flag_stop"])
    summ = q.assign(level=np.where(q["sgg_cd"] == SIDO_TOTAL_KEY, "시도", "시군구")) \
            .pivot_table(index="level", columns="grade", values="hs6", aggfunc="count", fill_value=0)
    print("[품질] 등급 분포 (계열 수)")
    display(summ)
    return q


# =============================================================================
# PART 8-7. [v4] 선택 예측: 품질 등급 통과 + (선택) 기업 연결 계열만
# =============================================================================
def select_forecast_targets() -> pd.DataFrame:
    with engine.connect() as conn:
        q = pd.read_sql(text(f"SELECT * FROM `{T_SERIES_QUAL}`"), conn)
    q = q[q["grade"].isin(FORECAST_GRADES)].copy()
    q["sgg_key"] = np.where(q["sgg_cd"] == SIDO_TOTAL_KEY, "", q["sgg_nm"].fillna(""))
    if FORECAST_LINKED_ONLY:
        cm = load_company_map()
        cm["hs6"] = cm["hs_code"].map(normalize_hs)
        cm["sgg_key"] = cm["sgg_nm"].fillna("")
        keys = set(zip(cm["sido_cd"].astype(str), cm["hs6"], cm["sgg_key"]))
        q = q[[k in keys for k in zip(q["sido_cd"].astype(str), q["hs6"], q["sgg_key"])]]
    try:
        with engine.connect() as conn:
            done = pd.read_sql(text(
                f"SELECT DISTINCT sido_cd, sgg_cd, hs_code FROM `{T_REGION_FORECAST}` "
                f"WHERE forecast_date = CURDATE()"), conn)
        done_keys = set(map(tuple, done.astype(str).values))
        q = q[[(s, (SIDO_TOTAL_KEY if k == "" else k[:40]), h) not in done_keys
               for s, k, h in zip(q["sido_cd"].astype(str), q["sgg_key"], q["hs6"])]]
    except Exception:
        pass
    q = q.sort_values(["grade", "avg_usd_year"], ascending=[True, False]).head(FORECAST_MAX_SERIES)
    return q[["sido_cd", "sgg_key", "hs6", "grade", "avg_usd_year"]].reset_index(drop=True)


def run_select_forecast() -> pd.DataFrame:
    tg = select_forecast_targets()
    print(f"[선택 예측] 대상 {len(tg)}계열 (등급 {FORECAST_GRADES}, 기업 연결만={FORECAST_LINKED_ONLY})")
    fd, log = datetime.today().strftime("%Y-%m-%d"), []
    for _, r in tg.iterrows():
        fr = forecast_and_save(r["sido_cd"], r["hs6"], r["sgg_key"], forecast_date=fd)
        log.append({"sido_cd": r["sido_cd"], "sgg_nm": r["sgg_key"], "hs6": r["hs6"], "grade": r["grade"],
                    "freq": fr.get("freq"), "n_obs": fr.get("n_obs"), "status": fr.get("status")})
    return pd.DataFrame(log)


## PART 7. 실행

7-1은 최초 1회(또는 API 오류 시) 실행합니다. 정상 확인 후 `RUN_PROBE = False`.

In [12]:
# =============================================================================
# PART 7-1. [최초 1회] API 응답 구조 확인
#   - resultCode가 정상('00')인지, 응답 필드명, 시군구 행/합계 행 구성, HS 계층을 확인
#   - 필드명이 FIELD_CANDIDATES와 다르면 PART 3-1 사전에 추가, 요청 파라미터명이 다르면 API_PARAM_NAMES 수정
# =============================================================================
RUN_PROBE = False
if RUN_PROBE:
    probe_df = probe_api(sido="제주", hs="854239", start="202401", end="202608")   # 12개월 초과도 자동 분할
    if not probe_df.empty:
        _std = standardize_raw(probe_df)
        _long, _diag = aggregate_region(_std, "854239")
        print("[probe] 표준화 진단:", _diag)
        display(_long[_long["indicator"] == "expDlr"].head(20))


In [13]:
# =============================================================================
# PART 7-2. 실행: 매핑 저장 -> 지역 수출 수집 -> 예측 -> 기업 매출 연동 분석
# =============================================================================
RUN_COMPANY_ANALYSIS = False   # True: 제주반도체 등 COMPANY_REGION_INPUTS 수집·예측·매출 연동 분석
if RUN_COMPANY_ANALYSIS:
    ensure_tables()
    company_map = upsert_company_map(COMPANY_REGION_INPUTS)
    display(company_map)

    # (1) 수집: (시도, HS) 단위 — 시군구 행은 같은 응답에 포함되어 함께 저장된다
    collect_log = []
    for (sd, hs), _ in company_map.groupby(["sido_cd", "hs_code"]):
        r = collect_region_hs(sd, hs)
        collect_log.append({"sido_cd": sd, "hs_code": hs, "rows_saved": r["rows_saved"], **(r["diag"] or {})})
        bad = r["meta"][r["meta"]["resultCode"].astype(str).isin(["00", "0", "None"]) == False] \
            if len(r["meta"]) else pd.DataFrame()
        if len(bad):
            print(f"[수집 경고] {sd}/{hs} 비정상 응답:")
            display(bad)
    display(pd.DataFrame(collect_log))

    # (2) 예측: (시도, 시군구, HS) 단위
    FORECAST_DATE = datetime.today().strftime("%Y-%m-%d")
    forecast_log = []
    for (sd, sgg, hs), _ in company_map.groupby(["sido_cd", "sgg_nm", "hs_code"]):
        fr = forecast_and_save(sd, hs, sgg, forecast_date=FORECAST_DATE)
        forecast_log.append({"sido_cd": sd, "sgg_nm": sgg, "hs_code": hs, "freq": fr.get("freq"),
                             "n_obs": fr.get("n_obs"), "zero_share": fr.get("zero_share"),
                             "status": fr.get("status")})
    display(pd.DataFrame(forecast_log))


In [16]:
# =============================================================================
# PART 7-3. 기업 매출 연동 분석 + 결과 저장
# =============================================================================
if RUN_COMPANY_ANALYSIS:
    fx_m = load_fx_monthly() if FX_ADJUST else None
    results = [analyze_company(r, fx_m) for _, r in company_map.iterrows()]
    # 같은 기업·지역에 HS가 2개 이상이면 합산 시계열 분석 추가
    for (tk, nm, sd, sgg), grp in company_map.groupby(["ticker", "company_name", "sido_cd", "sgg_nm"]):
        if grp["hs_code"].nunique() >= 2:
            results.append(analyze_company_group(tk, nm, sd, sgg, sorted(grp["hs_code"].unique()), fx_m))

    summary = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in results])
    pct_cols = [c for c in summary.columns if c.startswith("g_") or c in ("region_share_of_national_12m",)]
    show = summary.copy()
    for c in pct_cols:
        show[c] = (show[c] * 100).round(1)
    print("[요약] g_* = 향후 4분기 합계 / 최근 4분기(TTM) 합계 - 1 (%)")
    display(show)

    for r in results:
        plot_company(r)
        if r.get("status") == "ok":
            print(f"--- {r['company_name']} 분기 상세 (천원) ---")
            display(r["_detail"])
            print(f"--- {r['company_name']} 연동성 검정 (lag별) ---")
            display(r["_linkage"])

    out_xlsx = OUT_DIR / f"regional_export_revenue_{datetime.today():%Y%m%d}.xlsx"
    with pd.ExcelWriter(out_xlsx) as xw:
        show.to_excel(xw, sheet_name="Summary", index=False)
        det = [r["_detail"].reset_index().rename(columns={"index": "quarter"}) for r in results if r.get("status") == "ok"]
        if det:
            d = pd.concat(det)
            d["quarter"] = d["quarter"].astype(str)
            d.to_excel(xw, sheet_name="Quarterly_Detail", index=False)
        lk = [r["_linkage"] for r in results if r.get("status") == "ok"]
        if lk:
            pd.concat(lk).to_excel(xw, sheet_name="Linkage", index=False)
        pd.DataFrame(collect_log).to_excel(xw, sheet_name="Collect_Log", index=False)
        pd.DataFrame(forecast_log).to_excel(xw, sheet_name="Forecast_Log", index=False)
        pd.DataFrame({
            "item": ["forecast_date", "fx_adjust", "implied_horizon_q", "lags", "oos_start", "ensemble"],
            "value": [FORECAST_DATE, FX_ADJUST and fx_m is not None, IMPLIED_HORIZON_Q, str(LAGS), OOS_START, ENSEMBLE_AGG],
        }).to_excel(xw, sheet_name="Meta", index=False)
    print(f"[저장] {out_xlsx}")


## PART 8-8. [v4] 대량 수집 실행

In [17]:
# =============================================================================
# PART 8-8. [v4] 실행 (PART 8-0 의 스위치대로). 한도에 닿으면 멈추고, 다시 실행하면 이어서 받는다.
#   권장 순서: (1) 목록 -> (2) 1단계를 여러 날에 걸쳐 끝냄 -> (3) 2단계 -> (4) 품질 -> (5) 선택 예측
#   이후 매월: 3단계 -> 품질 -> 선택 예측
# =============================================================================
MANUAL_RUN = any([RUN_BUILD_UNIVERSE, RUN_STAGE1_SCAN, RUN_STAGE2_BACKFILL, RUN_STAGE3_UPDATE,
                  RUN_QUALITY, RUN_SELECT_FORECAST])
if MANUAL_RUN:
    ensure_tables()
    ensure_tables_v4()
    run_log = {}

    if RUN_BUILD_UNIVERSE:
        universe = build_hs_universe()
    else:
        universe = load_universe()
    print(f"[실행] 대상 HS6 {len(universe)}개")

    budget = CallBudget()
    if RUN_STAGE1_SCAN and budget.can_spend():
        run_log["stage1"] = run_stage1_scan(universe, budget)
    if RUN_STAGE2_BACKFILL and budget.can_spend():
        run_log["stage2"] = run_stage2_backfill(universe, budget)
    if RUN_STAGE3_UPDATE and budget.can_spend():
        run_log["stage3"] = run_stage3_update(budget)
    if RUN_QUALITY:
        run_log["quality"] = run_quality()
    if RUN_SELECT_FORECAST:
        run_log["forecast"] = run_select_forecast()

    # 진행 현황
    with engine.connect() as conn:
        scan_stat = pd.read_sql(text(
            f"SELECT status, COUNT(*) AS n, SUM(exp_12m_usd) AS usd FROM `{T_SCAN}` "
            f"WHERE scan_start = :s AND scan_end = :e GROUP BY status"), conn,
            params={"s": SCAN_START, "e": SCAN_END})
    total_combo = len(universe) * len(SCAN_SIDO_CODES)
    print(f"[현황] 1단계 {scan_stat['n'].sum() if len(scan_stat) else 0:,} / {total_combo:,} 조합 완료")
    display(scan_stat)
    print(f"[현황] 이번 실행 호출 {budget.used:,} | 중단 사유: {budget.stopped_reason or '없음'}")

    out_xlsx = OUT_DIR / f"regional_bulk_collect_{datetime.today():%Y%m%d_%H%M}.xlsx"
    with pd.ExcelWriter(out_xlsx) as xw:
        universe.to_excel(xw, sheet_name="Universe", index=False)
        scan_stat.to_excel(xw, sheet_name="Scan_Status", index=False)
        for k, v in run_log.items():
            if isinstance(v, pd.DataFrame) and len(v):
                v.head(100000).to_excel(xw, sheet_name=f"Log_{k}"[:31], index=False)
    print(f"[저장] {out_xlsx}")
else:
    print("[PART 8-8] 수동 스위치가 모두 False -> 건너뜀 (PART 10 자동 실행 사용)")


[PART 8-8] 수동 스위치가 모두 False -> 건너뜀 (PART 10 자동 실행 사용)


## PART 9. [v5] 지역 수출 예측 (품질 등급 기반, SARIMA 차수 적합 + ETS + Theta + 앙상블)

순서: PART 8 수집(1~2단계) -> `RUN_QUALITY=True` 로 품질 등급 -> `RUN_REGIONAL_FORECAST=True` 로 9-5 실행

In [18]:
# =============================================================================
# PART 9-0. [v5] 지역 수출 예측 설정 (품질 등급 기반, SARIMA 차수 적합 + ETS + Theta + 앙상블)
#   - 대상: korea_regional_series_quality 의 등급 A(월 예측), B(분기 예측)
#   - 저장: 새 테이블 2개 (넓은 형식) + 최신 예측 뷰
#       korea_regional_fc_run   : 계열 x 예측일 1행 (선택된 SARIMA 차수, 검증 오차, 가중치, 상태)
#       korea_regional_fc_value : 계열 x 예측일 x 대상월 1행 (ensemble, sarima, ets, theta 를 열로)
#       korea_regional_fc_latest: 계열별 가장 최근 예측일의 값만 보여 주는 뷰
#     기존 korea_regional_trade_forecast(긴 형식, 지표당 1행)는 PART 7 기업 분석용으로 그대로 둔다
# =============================================================================
RUN_REGIONAL_FORECAST = False      # PART 9-5 실행 스위치

FC_GRADES        = ["A", "B"]      # A: 월 예측, B: 분기 예측
FC_LEVELS        = ["sido", "sgg"] # 시도 합계, 시군구
FC_LINKED_ONLY   = False           # True: 지역 매핑(korea_company_region_hscode_map)에 있는 계열만
FC_PRIORITY_MAX  = 3               # 대상 HS 우선순위 상한 (1: 지역 매핑, 2: 기업 연결, 3: 전국 목록만)
FC_HORIZON_M     = 24              # 월 예측 기간 (분기 예측은 ceil(24/3)=8분기)
FC_MAX_SERIES    = None            # 한 번 실행에서 예측할 최대 계열 수 (None = 전부)
FC_SKIP_DONE     = True            # [v7] 같은 달에 이미 예측한 계열은 건너뜀
FC_N_JOBS        = 1               # 병렬 작업 수 (joblib). 4 정도 권장, 1 이면 순차
FC_SAVE_EVERY    = 100             # 이 계열 수마다 DB 저장

# SARIMA 차수 후보 (AICc 최소 선택). (p,d,q), (P,D,Q)
SARIMA_CANDIDATES = [
    ((0, 1, 1), (0, 1, 1)),   # airline 모형
    ((1, 1, 0), (0, 1, 1)),
    ((1, 1, 1), (0, 1, 1)),
    ((2, 1, 0), (0, 1, 1)),
    ((0, 1, 2), (0, 1, 1)),
    ((0, 1, 1), (1, 1, 0)),
    ((1, 0, 0), (0, 1, 1)),   # 상수 포함 (추세 있는 수준 계열)
    ((1, 1, 1), (1, 1, 1)),
]
FC_ENSEMBLE      = "median"        # "median" 또는 "inv_smape" (검증 오차 역수 가중)
FC_HOLDOUT       = {"M": 12, "Q": 4}   # 가중치·검증용 마지막 구간 길이 (월 12, 분기 4)
FC_BLOWOUT_X     = 5.0             # 앙상블 최댓값이 최근 최댓값의 이 배수를 넘으면 표시

T_FC_RUN    = "korea_regional_fc_run"
T_FC_VALUE  = "korea_regional_fc_value"
V_FC_LATEST = "korea_regional_fc_latest"


In [19]:
# =============================================================================
# PART 9-1. [v5] 예측 테이블 (넓은 형식) + 최신 예측 뷰
# =============================================================================
DDL_FC = {
    T_FC_RUN: f"""
        CREATE TABLE IF NOT EXISTS `{T_FC_RUN}` (
            `forecast_date` DATE NOT NULL,
            `sido_cd` VARCHAR(4) NOT NULL,
            `sgg_cd` VARCHAR(40) NOT NULL,
            `hs6` VARCHAR(6) NOT NULL,
            `freq` CHAR(1) NOT NULL,
            `sgg_nm` VARCHAR(60) NULL,
            `grade` CHAR(1) NULL,
            `n_obs` INT NULL,
            `last_actual` DATE NULL,
            `sarima_order` VARCHAR(20) NULL,
            `sarima_seasonal` VARCHAR(20) NULL,
            `sarima_aicc` DOUBLE NULL,
            `smape_sarima` DOUBLE NULL,
            `smape_ets` DOUBLE NULL,
            `smape_theta` DOUBLE NULL,
            `smape_ensemble` DOUBLE NULL,
            `smape_naive` DOUBLE NULL,
            `w_sarima` DOUBLE NULL,
            `w_ets` DOUBLE NULL,
            `w_theta` DOUBLE NULL,
            `ensemble_method` VARCHAR(12) NULL,
            `models_ok` VARCHAR(30) NULL,
            `status` VARCHAR(30) NOT NULL,
            `note` VARCHAR(200) NULL,
            `created_at` DATETIME DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (`forecast_date`, `sido_cd`, `sgg_cd`, `hs6`, `freq`),
            KEY `idx_series` (`hs6`, `sido_cd`, `sgg_cd`)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_general_ci""",
    T_FC_VALUE: f"""
        CREATE TABLE IF NOT EXISTS `{T_FC_VALUE}` (
            `forecast_date` DATE NOT NULL,
            `sido_cd` VARCHAR(4) NOT NULL,
            `sgg_cd` VARCHAR(40) NOT NULL,
            `hs6` VARCHAR(6) NOT NULL,
            `freq` CHAR(1) NOT NULL,
            `target_date` DATE NOT NULL,
            `ensemble` DOUBLE NULL,
            `sarima` DOUBLE NULL,
            `ets` DOUBLE NULL,
            `theta` DOUBLE NULL,
            PRIMARY KEY (`forecast_date`, `sido_cd`, `sgg_cd`, `hs6`, `freq`, `target_date`),
            KEY `idx_series` (`hs6`, `sido_cd`, `sgg_cd`, `forecast_date`)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_general_ci""",
}

VIEW_FC_LATEST = f"""
    CREATE OR REPLACE VIEW `{V_FC_LATEST}` AS
    SELECT v.*
    FROM `{T_FC_VALUE}` v
    JOIN (SELECT sido_cd, sgg_cd, hs6, freq, MAX(forecast_date) AS fd
          FROM `{T_FC_RUN}` WHERE status LIKE 'ok%%'
          GROUP BY sido_cd, sgg_cd, hs6, freq) m
      ON v.sido_cd = m.sido_cd AND v.sgg_cd = m.sgg_cd AND v.hs6 = m.hs6
     AND v.freq = m.freq AND v.forecast_date = m.fd"""


def ensure_tables_fc():
    if DRY_RUN_DB:
        print("[DRY_RUN] 예측 테이블 생성 생략")
        return
    with engine.begin() as conn:
        for ddl in DDL_FC.values():
            conn.execute(text(ddl))
        conn.execute(text(VIEW_FC_LATEST.replace("%%", "%")))
    print(f"[DB] 예측 테이블 확인/생성 완료: {list(DDL_FC)} + 뷰 {V_FC_LATEST}")


In [20]:
# =============================================================================
# PART 9-2. [v5] 예측 모형: SARIMA(후보 차수 중 AICc 최소) + ETS(감쇠 추세, 가법 계절) + Theta + 앙상블
#   - 로그 변환 z = log1p(수출액/1000) 에서 적합, 예측은 expm1 로 되돌려 0 미만은 0
#   - SARIMA 차수는 전체 자료로 고르고, 검증(마지막 FC_HOLDOUT 구간)은 그 차수로 다시 적합해 sMAPE 계산
#   - 검증에는 계절 단순 예측(작년 같은 달) sMAPE 도 함께 남겨, 모형이 단순 기준보다 나은지 볼 수 있게 한다
# =============================================================================
import warnings as _warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.forecasting.theta import ThetaModel


def _to_z(y):
    return np.log1p(np.clip(np.asarray(y, dtype=float), 0, None) / 1000.0)


def _from_z(z):
    return np.clip(np.expm1(np.asarray(z, dtype=float)) * 1000.0, 0, None)


def _smape(actual, pred) -> float:
    a, p = np.asarray(actual, float), np.asarray(pred, float)
    den = np.abs(a) + np.abs(p)
    ok = den > 0
    if not ok.any():
        return 0.0
    return float(np.mean(2.0 * np.abs(a[ok] - p[ok]) / den[ok]))


def _sarima_fit(z, m, order, sorder):
    trend = "c" if order[1] == 0 else "n"
    mod = SARIMAX(z, order=order, seasonal_order=(*sorder, m), trend=trend,
                  enforce_stationarity=False, enforce_invertibility=False)
    return mod.fit(disp=False, maxiter=60)


def select_sarima(z, m):
    """후보 차수 중 AICc 최소. 반환 (order, sorder, aicc, 적합결과) 또는 None"""
    best = None
    n = len(z)
    for order, sorder in SARIMA_CANDIDATES:
        if sorder[1] and n < 3 * m:          # 계절 차분에는 최소 3주기
            continue
        try:
            with _warnings.catch_warnings():
                _warnings.simplefilter("ignore")
                r = _sarima_fit(z, m, order, sorder)
            k = len(r.params)
            if n - k - 1 <= 0 or not np.isfinite(r.aic):
                continue
            aicc = r.aic + 2 * k * (k + 1) / (n - k - 1)
            if best is None or aicc < best[2]:
                best = (order, sorder, float(aicc), r)
        except Exception:
            continue
    return best


def _ets_forecast(z, m, h):
    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore")
        r = ExponentialSmoothing(z, trend="add", damped_trend=True, seasonal="add",
                                 seasonal_periods=m, initialization_method="estimated").fit()
    return np.asarray(r.forecast(h))


def _theta_forecast(z, m, h):
    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore")
        r = ThetaModel(np.asarray(z), period=m, deseasonalize=True, method="additive").fit()
    return np.asarray(r.forecast(h))


def _run_models(z, m, h, sarima_spec=None):
    """z 로 h 기간 예측. sarima_spec=(order, sorder) 면 그 차수, None 이면 후보 탐색. 반환 dict, spec"""
    out, spec = {}, None
    try:
        if sarima_spec is None:
            sel = select_sarima(z, m)
            if sel is not None:
                spec = (sel[0], sel[1], sel[2])
                out["sarima"] = np.asarray(sel[3].forecast(h))
        else:
            with _warnings.catch_warnings():
                _warnings.simplefilter("ignore")
                r = _sarima_fit(z, m, sarima_spec[0], sarima_spec[1])
            out["sarima"] = np.asarray(r.forecast(h))
    except Exception:
        pass
    for name, fn in (("ets", _ets_forecast), ("theta", _theta_forecast)):
        try:
            if len(z) >= 2 * m + 2:
                fc = fn(z, m, h)
                if np.all(np.isfinite(fc)):
                    out[name] = fc
        except Exception:
            pass
    return {k: _from_z(v) for k, v in out.items() if np.all(np.isfinite(v))}, spec


def _combine(fcs: dict, weights: dict = None):
    names = [k for k in ("sarima", "ets", "theta") if k in fcs]
    arr = np.vstack([fcs[k] for k in names])
    if weights and all(weights.get(k) is not None for k in names):
        w = np.array([weights[k] for k in names], float)
        w = w / w.sum()
        return (arr * w[:, None]).sum(axis=0)
    return np.median(arr, axis=0)


def forecast_one_series(y: pd.Series, freq: str) -> dict:
    """y: 0 채움된 연속 계열 (월말 또는 분기말 인덱스). 반환 dict(meta, values DataFrame)"""
    m = 12 if freq == "M" else 4
    h = FC_HORIZON_M if freq == "M" else int(math.ceil(FC_HORIZON_M / 3))
    H = FC_HOLDOUT[freq]
    y = y.astype(float)
    z = _to_z(y.values)
    meta = {"n_obs": int(len(y)), "last_actual": y.index.max().date(), "freq": freq,
            "sarima_order": None, "sarima_seasonal": None, "sarima_aicc": None,
            "smape_sarima": None, "smape_ets": None, "smape_theta": None, "smape_ensemble": None,
            "smape_naive": None, "w_sarima": None, "w_ets": None, "w_theta": None,
            "ensemble_method": FC_ENSEMBLE, "models_ok": "", "status": "", "note": ""}
    # (1) 전체 자료: SARIMA 차수 선택 + 세 모형 예측
    fcs, spec = _run_models(z, m, h)
    if not fcs:
        meta["status"] = "failed"
        meta["note"] = "모든 모형 실패"
        return {"meta": meta, "values": None}
    if spec:
        meta["sarima_order"] = str(spec[0]).replace(" ", "")
        meta["sarima_seasonal"] = str(spec[1]).replace(" ", "")
        meta["sarima_aicc"] = spec[2]
    # (2) 검증: 마지막 H 구간을 떼고 같은 차수로 다시 적합
    weights = None
    if len(y) > H + 3 * m:
        hfcs, _ = _run_models(z[:-H], m, H, sarima_spec=(spec[0], spec[1]) if spec else None)
        actual = y.values[-H:]
        sm = {k: _smape(actual, v) for k, v in hfcs.items()}
        for k, v in sm.items():
            meta[f"smape_{k}"] = v
        meta["smape_naive"] = _smape(actual, y.values[-H - m:-m])
        if hfcs:
            inv = {k: 1.0 / max(v, 1e-6) for k, v in sm.items()}
            if FC_ENSEMBLE == "inv_smape" and set(inv) >= set(fcs):
                weights = inv
                tot = sum(inv[k] for k in fcs)
                for k in fcs:
                    meta[f"w_{k}"] = inv[k] / tot
            meta["smape_ensemble"] = _smape(actual, _combine(hfcs, weights))
    ens = _combine(fcs, weights)
    if weights is None:
        meta["ensemble_method"] = "median"
    meta["models_ok"] = ",".join(k for k in ("sarima", "ets", "theta") if k in fcs)
    recent_max = float(y.tail(2 * m).max())
    status = "ok" if len(fcs) == 3 else "ok_partial"
    if recent_max > 0 and float(np.max(ens)) > FC_BLOWOUT_X * recent_max:
        status += "_blowout"
    meta["status"] = status
    off = pd.offsets.MonthEnd(1) if freq == "M" else pd.offsets.QuarterEnd(1)
    idx = pd.date_range(y.index.max() + off, periods=h, freq=FREQ_M if freq == "M" else FREQ_Q)
    vals = pd.DataFrame({"target_date": idx.date, "ensemble": ens,
                         "sarima": fcs.get("sarima", [np.nan] * h),
                         "ets": fcs.get("ets", [np.nan] * h),
                         "theta": fcs.get("theta", [np.nan] * h)})
    return {"meta": meta, "values": vals}


In [21]:
# =============================================================================
# PART 9-3. [v5] 예측 대상 선택 + 계열 만들기 (HS 단위로 한 번에 읽음)
#   - 품질 등급과 같은 규칙: 수집 범위 안 빈 달 = 0, 첫 수출 달 이후만 사용
#   - A: 월 계열, B: 완결된 분기 합계 계열
#   - 통합코드 12: 2026-06 까지는 29+46, 2026-07 부터 12 (품질 계산과 같은 _to_target_sido)
# =============================================================================
def select_fc_targets(forecast_date: str) -> pd.DataFrame:
    with engine.connect() as conn:
        from sqlalchemy import bindparam
        q = pd.read_sql(text(f"SELECT sido_cd, sgg_cd, hs6, sgg_nm, grade FROM `{T_SERIES_QUAL}` "
                             f"WHERE grade IN :g").bindparams(bindparam("g", expanding=True)),
                        conn, params={"g": list(FC_GRADES)})
        uni = pd.read_sql(text(f"SELECT hs6, priority FROM `{T_HS_UNIVERSE}`"), conn)
    if q.empty:
        return q
    q["level"] = np.where(q["sgg_cd"] == SIDO_TOTAL_KEY, "sido", "sgg")
    q = q[q["level"].isin(FC_LEVELS)]
    q = q.merge(uni, on="hs6", how="left")
    q["priority"] = q["priority"].fillna(3).astype(int)
    q = q[q["priority"] <= FC_PRIORITY_MAX]
    q["sgg_key"] = np.where(q["level"] == "sido", "", q["sgg_nm"].fillna(""))
    if FC_LINKED_ONLY:
        cm = load_company_map()
        cm["hs6"] = cm["hs_code"].map(normalize_hs)
        keys = set(zip(cm["sido_cd"].astype(str), cm["hs6"], cm["sgg_nm"].fillna("")))
        q = q[[k in keys for k in zip(q["sido_cd"].astype(str), q["hs6"], q["sgg_key"])]]
    if FC_SKIP_DONE:
        try:
            with engine.connect() as conn:
                fdt = pd.Timestamp(forecast_date)
                done = pd.read_sql(text(f"SELECT sido_cd, sgg_cd, hs6 FROM `{T_FC_RUN}` "
                                        f"WHERE YEAR(forecast_date) = :y AND MONTH(forecast_date) = :m"),
                                   conn, params={"y": fdt.year, "m": fdt.month})   # [v7] 같은 달 예측은 다시 안 함
            dk = set(map(tuple, done.astype(str).values))
            q = q[[(a, b, c) not in dk for a, b, c in zip(q["sido_cd"].astype(str), q["sgg_cd"].astype(str), q["hs6"])]]
        except Exception:
            pass
    q = q.sort_values(["priority", "grade", "hs6", "sido_cd", "sgg_cd"]).reset_index(drop=True)
    if FC_MAX_SERIES:
        q = q.head(int(FC_MAX_SERIES))
    return q


def build_series_for_hs(hs: str, targets: pd.DataFrame, cov: pd.DataFrame) -> list:
    """한 HS 의 예측 대상 계열들을 만든다. 반환 [(key dict, series, freq)]"""
    with engine.connect() as conn:
        d = pd.read_sql(text(f"SELECT `date`, sido_cd, sgg_cd, sgg_nm, value FROM `{T_REGION_TRADE}` "
                             f"WHERE hs_code = :hs AND indicator = 'expDlr'"), conn, params={"hs": hs})
    if d.empty:
        return []
    d["date"] = pd.to_datetime(d["date"]) + pd.offsets.MonthEnd(0)
    d["target"] = d["sido_cd"].astype(str).map(_to_target_sido)
    out = []
    for _, t in targets.iterrows():
        c = cov[(cov["target"] == t["sido_cd"]) & (cov["hs6"] == hs)]
        if c.empty:
            continue
        c = c.iloc[0]
        c_start = pd.Timestamp(c["c_start"][:4] + "-" + c["c_start"][4:] + "-01") + pd.offsets.MonthEnd(0)
        c_end = pd.Timestamp(c["c_end"][:4] + "-" + c["c_end"][4:] + "-01") + pd.offsets.MonthEnd(0)
        sub = d[d["target"] == t["sido_cd"]]
        sub = sub[sub["sgg_cd"] == SIDO_TOTAL_KEY] if t["sgg_cd"] == SIDO_TOTAL_KEY else sub[sub["sgg_nm"] == t["sgg_key"]]
        s = sub.groupby("date")["value"].sum()
        s = s.reindex(pd.date_range(c_start, c_end, freq=FREQ_M)).fillna(0.0)
        pos = s[s > 0]
        if pos.empty:
            continue
        s = s[s.index >= pos.index.min()]
        if t["grade"] == "B":
            q = s.resample(FREQ_Q).sum()
            if s.index.max() != s.index.max() + pd.offsets.QuarterEnd(0):
                q = q.iloc[:-1]
            s, freq = q, "Q"
        else:
            freq = "M"
        key = {"sido_cd": t["sido_cd"], "sgg_cd": t["sgg_cd"], "hs6": hs,
               "sgg_nm": (t["sgg_nm"] or "")[:60] if t["sgg_cd"] != SIDO_TOTAL_KEY else "", "grade": t["grade"]}
        out.append((key, s, freq))
    return out


In [22]:
# =============================================================================
# PART 9-4. [v5] 예측 실행 + 저장 (FC_SAVE_EVERY 계열마다 저장, 중단돼도 끝난 계열은 남음)
# =============================================================================
def _task(key, s, freq):
    try:
        r = forecast_one_series(s, freq)
    except Exception as e:
        r = {"meta": {"status": "failed", "note": str(e)[:200], "n_obs": len(s), "freq": freq,
                      "last_actual": s.index.max().date() if len(s) else None}, "values": None}
    return key, r


def _flush(runs: list, values: list) -> None:
    if runs:
        rdf = pd.DataFrame(runs)
        upsert_rows(T_FC_RUN, rdf, key_cols=["forecast_date", "sido_cd", "sgg_cd", "hs6", "freq"],
                    upd_cols=[c for c in rdf.columns if c not in ("forecast_date", "sido_cd", "sgg_cd", "hs6", "freq")])
    if values:
        vdf = pd.concat(values, ignore_index=True)
        upsert_rows(T_FC_VALUE, vdf, key_cols=["forecast_date", "sido_cd", "sgg_cd", "hs6", "freq", "target_date"],
                    upd_cols=["ensemble", "sarima", "ets", "theta"])
    runs.clear()
    values.clear()


RUN_COLS = ["sgg_nm", "grade", "n_obs", "last_actual", "sarima_order", "sarima_seasonal", "sarima_aicc",
            "smape_sarima", "smape_ets", "smape_theta", "smape_ensemble", "smape_naive",
            "w_sarima", "w_ets", "w_theta", "ensemble_method", "models_ok", "status", "note"]


def run_regional_forecast(forecast_date: Optional[str] = None) -> pd.DataFrame:
    fd = forecast_date or datetime.today().strftime("%Y-%m-%d")
    ensure_tables_fc()
    targets = select_fc_targets(fd)
    if targets.empty:
        print("[지역 예측] 대상 없음 (품질 등급 A/B 계열이 없거나 모두 완료)")
        return pd.DataFrame()
    cov = _coverage_by_target()
    print(f"[지역 예측] 예측일 {fd} | 대상 {len(targets):,}계열 "
          f"(A {int((targets.grade=='A').sum())}, B {int((targets.grade=='B').sum())}) | HS {targets.hs6.nunique()}개 "
          f"| 병렬 {FC_N_JOBS} | 앙상블 {FC_ENSEMBLE}")
    runs, values, log, t0, n_done = [], [], [], time.time(), 0
    for hs, tg in targets.groupby("hs6", sort=False):
        series = build_series_for_hs(hs, tg, cov)
        if not series:
            continue
        if FC_N_JOBS and FC_N_JOBS > 1 and len(series) > 1:
            from joblib import Parallel, delayed
            results = Parallel(n_jobs=FC_N_JOBS)(delayed(_task)(k, s, f) for k, s, f in series)
        else:
            results = [_task(k, s, f) for k, s, f in series]
        for key, r in results:
            meta = r["meta"]
            row = {"forecast_date": fd, "sido_cd": key["sido_cd"], "sgg_cd": key["sgg_cd"], "hs6": key["hs6"],
                   "freq": meta.get("freq"), "sgg_nm": key["sgg_nm"], "grade": key["grade"]}
            row.update({c: meta.get(c) for c in RUN_COLS if c not in ("sgg_nm", "grade")})
            runs.append(row)
            if r["values"] is not None:
                v = r["values"].copy()
                v["forecast_date"], v["sido_cd"], v["sgg_cd"] = fd, key["sido_cd"], key["sgg_cd"]
                v["hs6"], v["freq"] = key["hs6"], meta.get("freq")
                values.append(v)
            log.append(row)
            n_done += 1
        if len(runs) >= FC_SAVE_EVERY:
            _flush(runs, values)
            el = time.time() - t0
            print(f"   ... {n_done:,}/{len(targets):,} 계열 | {el/60:.1f}분 | 계열당 {el/max(n_done,1):.1f}초")
    _flush(runs, values)
    lg = pd.DataFrame(log)
    if len(lg):
        print(f"[지역 예측] 완료 {len(lg):,}계열 | {(time.time()-t0)/60:.1f}분")
        print(lg["status"].value_counts().to_string())
        okm = lg[lg["status"].str.startswith("ok")]
        if len(okm) and okm["smape_naive"].notna().any():
            cmp = okm.dropna(subset=["smape_ensemble", "smape_naive"])
            if len(cmp):
                print(f"[검증] 앙상블 sMAPE 중앙값 {cmp.smape_ensemble.median():.3f} vs 작년 같은 달 {cmp.smape_naive.median():.3f} "
                      f"| 앙상블이 더 나은 계열 {(cmp.smape_ensemble < cmp.smape_naive).mean():.0%}")
    return lg


In [23]:
# =============================================================================
# PART 9-5. [v5] 지역 수출 예측 실행 (PART 8 수집 -> 품질 등급(RUN_QUALITY) 이 먼저 끝나 있어야 함)
#   최신 예측 조회 예:  SELECT * FROM korea_regional_fc_latest WHERE hs6='854232' AND sido_cd='50'
# =============================================================================
if RUN_REGIONAL_FORECAST:
    fc_log = run_regional_forecast()
    if len(fc_log):
        p = OUT_DIR / f"regional_forecast_log_{datetime.today():%Y%m%d_%H%M}.csv"
        fc_log.to_csv(p, index=False, encoding="utf-8-sig")
        print(f"[저장] {p}")
else:
    print("[PART 9] RUN_REGIONAL_FORECAST=False -> 건너뜀")


[PART 9] RUN_REGIONAL_FORECAST=False -> 건너뜀


## PART 10. [v6] 자동 실행 (매일 이 노트북을 전체 실행)

In [25]:
# =============================================================================
# PART 10. [v7] 자동 실행 -- 이 노트북은 "전체 실행" 한 번이면 된다. 매일 한 번 실행하면 다음 일을 알아서 한다.
#   (1) 대상 HS 목록 갱신 (API 호출 없음, 몇 초)
#   (2) 1단계(수출 지도)가 안 끝났으면 1단계 -> 끝났으면 2단계(과거 채우기) -> 끝났으면 월 1회 3단계(갱신)
#       하루 호출 한도(DAILY_CALL_LIMIT)에 닿으면 멈춘다. 내일 다시 실행하면 이어서 받는다.
#   (3) 새로 받은 자료가 있으면 품질 등급 다시 계산
#   (4) 과거 자료가 다 찬 계열(A/B)부터 매일 예측. 같은 달에 이미 예측한 계열은 건너뜀 (PART 9)
#   (5) 진행 현황을 한눈에 출력
# =============================================================================
AUTO_RUN = True          # False 로 두면 자동 실행 안 함 (PART 8-8 / 9-5 수동 스위치 방식)


def stage1_remaining(universe: pd.DataFrame) -> Tuple[int, int]:
    done = _scan_done_keys()
    total = len(universe) * len(SCAN_SIDO_CODES)
    left = sum(1 for hs in universe["hs6"] for sd in SCAN_SIDO_CODES if (sd, hs) not in done)
    return left, total


def stage2_remaining(universe: pd.DataFrame) -> Tuple[int, int]:
    cand = stage2_candidates(universe)
    if cand.empty:
        return 0, 0
    hist_end = int_to_yymm(yymm_to_int(SCAN_START) - 1)
    done = load_done_windows()
    total = left = 0
    for _, r in cand.iterrows():
        for req_cd, s, e in plan_requests(str(r["sido_cd"]), START_YYYYMM, hist_end):
            for ws, we in split_windows(s, e):
                total += 1
                if (req_cd, r["hs6"], ws) not in done:
                    left += 1
    return left, total


def _last_stage3_date():
    try:
        with engine.connect() as conn:
            d = pd.read_sql(text(f"SELECT MAX(updated_at) AS d FROM `{T_COLLECT_STAT}` WHERE stage = 3"), conn)["d"].iloc[0]
        return pd.Timestamp(d) if pd.notna(d) else None
    except Exception:
        return None


def _forecast_done_this_month() -> bool:
    try:
        with engine.connect() as conn:
            d = pd.read_sql(text(f"SELECT MAX(forecast_date) AS d FROM `{T_FC_RUN}`"), conn)["d"].iloc[0]
        return pd.notna(d) and pd.Timestamp(d).strftime("%Y%m") == datetime.today().strftime("%Y%m")
    except Exception:
        return False


def auto_run():
    print("=" * 70)
    print(f"[자동 실행] {datetime.today():%Y-%m-%d %H:%M}")
    print("=" * 70)
    ensure_tables()
    ensure_tables_v4()
    ensure_tables_fc()

    universe = build_hs_universe()
    if universe is None or universe.empty:
        print("[자동 실행] 대상 HS 목록이 비어 있습니다. 출처 설정을 확인하세요.")
        return
    budget = CallBudget()
    collected = False

    # (2) 수집
    s1_left, s1_total = stage1_remaining(universe)
    if s1_left > 0:
        print(f"\n[단계] 1단계 수출 지도: 남은 조합 {s1_left:,} / {s1_total:,}")
        if budget.can_spend():
            run_stage1_scan(universe, budget)
            collected = True
        s1_left, _ = stage1_remaining(universe)
    if s1_left == 0:
        s2_left, s2_total = stage2_remaining(universe)
        if s2_left > 0:
            print(f"\n[단계] 2단계 과거 채우기: 남은 구간 {s2_left:,} / {s2_total:,}")
            if budget.can_spend():
                run_stage2_backfill(universe, budget)
                collected = True
            s2_left, _ = stage2_remaining(universe)
        if s2_left == 0:
            last3 = _last_stage3_date()
            if last3 is None or (pd.Timestamp.today() - last3).days >= 25:
                print("\n[단계] 3단계 월간 갱신")
                if budget.can_spend():
                    run_stage3_update(budget)
                    collected = True
    else:
        s2_left, s2_total = None, None

    # (3) 품질 등급
    try:
        with engine.connect() as conn:
            n_q = pd.read_sql(text(f"SELECT COUNT(*) AS n FROM `{T_SERIES_QUAL}`"), conn)["n"].iloc[0]
    except Exception:
        n_q = 0
    if collected or n_q == 0:
        print("\n[단계] 품질 등급 계산")
        run_quality()

    # (4) [v7] 예측: 과거 자료가 다 찬 계열(품질 A/B) 중 이번 달 아직 예측하지 않은 것
    print("\n[단계] 지역 수출 예측 (과거 자료가 다 찬 A/B 계열만)")
    lg = run_regional_forecast()
    try:
        with engine.connect() as conn:
            gq = pd.read_sql(text(f"SELECT grade, COUNT(*) AS n FROM `{T_SERIES_QUAL}` GROUP BY grade"), conn)
            fdt = pd.Timestamp.today()
            nfc = pd.read_sql(text(f"SELECT COUNT(*) AS n FROM `{T_FC_RUN}` WHERE YEAR(forecast_date) = :y "
                                   f"AND MONTH(forecast_date) = :m AND status LIKE 'ok%%'"),
                              conn, params={"y": fdt.year, "m": fdt.month})["n"].iloc[0]
        gd = dict(zip(gq["grade"], gq["n"]))
        fc_msg = (f"이번 달 예측 완료 {int(nfc):,}계열 (이번 실행 {len(lg):,}) | 등급 A {gd.get('A',0):,}, B {gd.get('B',0):,}, "
                  f"수집 미완 P {gd.get('P',0):,}, C {gd.get('C',0):,}, X {gd.get('X',0):,}")
    except Exception:
        fc_msg = f"이번 실행 {len(lg):,}계열 예측"

    # (5) 현황
    s1_left, s1_total = stage1_remaining(universe)
    s2_txt = "-"
    if s1_left == 0:
        s2_left, s2_total = stage2_remaining(universe)
        s2_txt = f"{s2_total - s2_left:,} / {s2_total:,} 구간" if s2_total else "대상 없음"
    print("\n" + "=" * 70)
    print("[진행 현황]")
    print(f"  대상 HS6           : {len(universe):,}개")
    print(f"  1단계 수출 지도    : {s1_total - s1_left:,} / {s1_total:,} 조합 ({(s1_total - s1_left) / max(s1_total, 1):.0%})")
    print(f"  2단계 과거 채우기  : {s2_txt}")
    print(f"  예측               : {fc_msg}")
    print(f"  오늘 호출          : 이번 실행 {budget.used:,}회 | 중단 사유: {budget.stopped_reason or '없음'}")
    if budget.stopped_reason:
        print("  -> 오늘 한도를 다 썼습니다. 내일 이 노트북을 다시 '전체 실행' 하면 이어서 받습니다.")
    print("=" * 70)


if AUTO_RUN:
    auto_run()


[자동 실행] 2026-10-25 23:07
[DB] 테이블 확인/생성 완료: ['korea_regional_trade_data', 'korea_regional_trade_forecast', 'korea_company_region_hscode_map']
[DB] v4 테이블 확인/생성 완료: ['korea_regional_hs_universe', 'korea_regional_trade_scan', 'korea_regional_collect_status', 'korea_regional_series_quality']
[DB] 예측 테이블 확인/생성 완료: ['korea_regional_fc_run', 'korea_regional_fc_value'] + 뷰 korea_regional_fc_latest
[universe] HS_CODE_MAP 코드 546개 (예: ['0207', '030341', '030342', '030343', '030344'])
[universe] kr_company_hs_map: 코드 501개 (HS 열 'hs_code', 기업 열 'ticker')
[universe] fd_driver_map_2d: 수출 코드 24개
[universe] korea_company_region_hscode_map: 코드 3개, 매핑 3행
[universe] 6자리 참조 사전 5629개
[universe] 6자리로 펼치지 못한 4자리 코드 2개 -> C:\Users\82108\OneDrive\INVESTMENT\한국_지역별수출분석\202610\hs_universe_unresolved_20261025.csv (HS6_MASTER_CSV 로 보충 가능)
[universe] 대상 HS6 1123개 (priority 1: 3, 2: 858, 3: 262) -> 1단계 예상 호출 약 17,968회 (통합코드 12 는 구간에 따라 최대 3회)
[예산] 오늘 이미 사용 0 | 이번 실행 가능 49800

[단계] 2단계 과거 채우기: 남은 구간 45,609 / 104,535


KeyboardInterrupt: 

In [49]:
# with engine.connect() as conn:
#     r = pd.read_sql(text("SELECT freq, smape_ensemble, smape_naive, smape_sarima, smape_ets, smape_theta "
#                          "FROM korea_regional_fc_run WHERE status LIKE 'ok%%'"), conn)
# r = r.dropna(subset=["smape_ensemble", "smape_naive"])
# print(f"계열 {len(r):,}개")
# print(r.groupby("freq")[["smape_ensemble", "smape_naive", "smape_sarima", "smape_ets", "smape_theta"]].median().round(3))
# print(f"앙상블이 작년 같은 달보다 나은 계열: {(r.smape_ensemble < r.smape_naive).mean():.0%}")

In [50]:
# with engine.connect() as conn:
#     g = pd.read_sql(text("SELECT grade, COUNT(*) AS n FROM korea_regional_series_quality GROUP BY grade"), conn)
#     r = pd.read_sql(text("SELECT freq, status, smape_ensemble, smape_naive FROM korea_regional_fc_run"), conn)
# print("품질 등급별 계열 수"); print(g)
# print("\n예측 상태"); print(r["status"].value_counts())
# ok = r[r["status"].str.startswith("ok")].dropna(subset=["smape_ensemble", "smape_naive"])
# print(f"\n검증 가능 계열 {len(ok):,}개")
# print(ok.groupby("freq")[["smape_ensemble", "smape_naive"]].median().round(3))
# print(f"앙상블이 작년 같은 달보다 나은 계열: {(ok.smape_ensemble < ok.smape_naive).mean():.0%}")

In [51]:
# import numpy as np
# import pandas as pd
# from sqlalchemy import text
#
# with engine.connect() as conn:
#     r = pd.read_sql(text("""
#         SELECT f.freq, f.sido_cd, f.sgg_cd, q.sgg_nm, f.smape_ensemble, f.smape_naive, q.avg_usd_year
#         FROM korea_regional_fc_run f
#         JOIN korea_regional_series_quality q
#           ON f.sido_cd = q.sido_cd AND f.sgg_cd = q.sgg_cd AND f.hs6 = q.hs6
#         WHERE f.status LIKE 'ok%%'"""), conn)
# r = r.dropna(subset=["smape_ensemble", "smape_naive"])
# r["단위"] = np.where(r["sgg_nm"].fillna("").str.strip() == "", "시도", "시군구")
# r["규모"] = pd.cut(r["avg_usd_year"], [0, 5e6, 2e7, 1e8, np.inf],
#                    labels=["~500만$", "500만~2천만$", "2천만~1억$", "1억$~"])
# r["나음"] = r["smape_ensemble"] < r["smape_naive"]
# t = r.groupby(["freq", "단위", "규모"], observed=True).agg(
#     계열=("smape_ensemble", "size"),
#     앙상블=("smape_ensemble", "median"),
#     작년같은달=("smape_naive", "median"),
#     나은비율=("나음", "mean"))
# print(t.round(3))